# EDM 论文完整详解笔记（优化版）

**论文：** *Elucidating the Design Space of Diffusion-Based Generative Models*  
**作者：** Tero Karras 等  
**会议：** NeurIPS 2022  
**论文：** https://arxiv.org/abs/2206.00364  
**代码：** https://github.com/NVlabs/edm


## 一、论文的核心思想与主要结论

### 1.1 EDM 究竟提出了什么？

EDM 并不是提出了一种全新的扩散过程，也不能简单理解为：

$$
\text{EDM}=\text{VE SDE}+\text{Heun Solver}
$$

它的核心贡献是将扩散生成模型拆分为多个可以相对独立优化的模块：

$$
\boxed{
\text{Diffusion Model}
=
\text{Distribution}
+
\text{Sampling}
+
\text{Network}
+
\text{Training}
}
\tag{1.1}
$$

其中：

**分布层面：** 统一描述 VP、VE 等不同的加噪过程。

**采样层面：** 研究 ODE 参数化、数值积分器、时间网格和随机扰动。

**网络层面：** 研究 Denoiser 的输入、输出及噪声条件的预条件化。

**训练层面：** 研究训练噪声分布、Loss 权重、网络容量和数据增强。

EDM 的研究思路是：在尽可能控制其他变量的情况下，分别改进这些设计，并通过实验判断它们对生成质量和计算效率的影响。

### 1.2 六个重要结论

**结论一：VP 与 VE 可以放入统一的数学框架。**

引入两个函数：

$$
\boxed{
s(t),\qquad \sigma(t)
}
\tag{1.2}
$$

其中：

- $s(t)$：对信号和噪声共同施加的状态缩放系数。
- $\sigma(t)$：在未缩放坐标下定义的噪声标准差。

通过它们可以表示不同扩散模型的边缘分布。

**结论二：推荐采用简单的 ODE 参数化。**

EDM 的推荐选择为：

$$
\boxed{
s(t)=1,\qquad \sigma(t)=t
}
\tag{1.3}
$$

此时 Probability Flow ODE 简化为：

$$
\boxed{
\frac{dx}{dt}
=
\frac{x-D_\theta(x;t)}{t}
}
\tag{1.4}
$$

这里 $D_\theta(x;t)$ 是去噪网络。

需要注意，$t$ 在这个公式中直接表示噪声标准差，而不是必须位于 $[0,1]$ 的扩散时间。

**结论三：Heun 与非均匀噪声网格可以提高采样效率。**

在相同或相近的 NFE 下，二阶 Heun 通常比一阶 Euler 具有更小的数值积分误差。

同时，通过合理安排不同噪声区间的步长，可以减少不必要的网络计算。

**结论四：适量随机性可能改善生成质量。**

EDM 通过 Stochastic Churn 在采样过程中重新注入少量噪声，再使用确定性 ODE 去噪。

这可以在某些设置下减轻采样误差造成的分布偏差，但并不意味着随机性越大越好。

**结论五：网络预条件化能够改善不同噪声水平下的优化条件。**

EDM 使用：

$$
\boxed{
D_\theta(x;\sigma)
=
c_{\mathrm{skip}}(\sigma)x
+
c_{\mathrm{out}}(\sigma)
F_\theta
\left(
c_{\mathrm{in}}(\sigma)x,
c_{\mathrm{noise}}(\sigma)
\right)
}
\tag{1.5}
$$

其目的不是改变最优 Denoiser 的数学定义，而是让神经网络在不同噪声水平下更容易学习相应的映射。

**结论六：噪声采样分布与 Loss 权重应当联合设计。**

EDM 推荐：

$$
\boxed{
\ln\sigma
\sim
\mathcal N(P_{\mathrm{mean}},P_{\mathrm{std}}^2)
}
\tag{1.6}
$$

以及：

$$
\boxed{
\lambda(\sigma)
=
\frac{\sigma^2+\sigma_{\mathrm{data}}^2}
{\sigma^2\sigma_{\mathrm{data}}^2}
}
\tag{1.7}
$$

它们分别决定训练时不同噪声水平被采样的频率，以及每次采样对优化目标的贡献。

---

## 二、Section 1：Introduction

### 2.1 为什么需要 EDM？

EDM 之前已经存在多种扩散模型：

- DDPM
- DDIM
- VP SDE
- VE SDE
- Score-Based Generative Models
- Improved DDPM

这些模型可能使用不同的加噪过程。

例如，VP 类模型常使用：

$$
x_t=\alpha(t)x_0+\bar\sigma(t)z
\tag{2.1}
$$

而 VE 类模型常使用：

$$
x_t=x_0+\sigma(t)z
\tag{2.2}
$$

网络预测目标也可能不同：

$$
\epsilon_\theta(x_t,t)
$$

$$
s_\theta(x_t,t)
$$

$$
D_\theta(x_t,t)
$$

分别对应噪声预测、Score 预测和干净数据预测。

除此之外，不同模型还使用不同的时间离散化方法和采样器。

于是出现一个问题：

**当两个扩散模型的生成质量不同，我们很难直接判断差异究竟来自加噪过程、网络结构、训练目标还是采样器。**

EDM 因此选择首先建立统一数学框架，然后分别研究各模块。

### 2.2 论文的整体研究路线

原论文的主要结构为：

| 章节 | 研究内容 | 核心问题 |
|---|---|---|
| Section 2 | Common Framework | 如何统一 VP、VE 等模型？ |
| Section 3 | Deterministic Sampling | 如何更高效地求解生成 ODE？ |
| Section 4 | Stochastic Sampling | 为什么以及如何引入随机性？ |
| Section 5 | Training | 如何改善网络训练？ |
| Section 6 | Discussion | 这些设计带来什么启示？ |

下面先从最基本的概率分布开始推导。

---

## 三、Section 2：Expressing Diffusion Models in a Common Framework

### 3.1 定义高斯加噪分布

首先定义干净数据：

$$
\boxed{
Y\sim p_{\mathrm{data}}(y)
}
\tag{3.1}
$$

其中：

- $Y\in\mathbb R^d$：随机的真实数据。
- $p_{\mathrm{data}}$：真实数据分布。
- $d$：数据维度。

定义标准高斯噪声：

$$
Z\sim\mathcal N(0,I)
\tag{3.2}
$$

其中 $Z$ 与 $Y$ 独立。

对于噪声标准差 $\sigma>0$，构造：

$$
\boxed{
X_\sigma=Y+\sigma Z
}
\tag{3.3}
$$

给定 $Y=y$，加噪状态满足：

$$
\boxed{
p(x\mid y;\sigma)
=
\mathcal N(x;y,\sigma^2I)
}
\tag{3.4}
$$

将干净数据 $Y$ 积分掉，就得到加噪后的边缘分布：

$$
\boxed{
p_\sigma(x)
=
\int
p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)
\,dy
}
\tag{3.5}
$$

这就是数据分布与高斯分布的卷积。

当：

$$
\sigma\rightarrow0
$$

有：

$$
p_\sigma(x)\rightarrow p_{\mathrm{data}}(x)
\tag{3.6}
$$

当：

$$
\sigma\gg\sigma_{\mathrm{data}}
$$

并且数据已经中心化时：

$$
p_\sigma(x)
\approx
\mathcal N(0,\sigma^2I)
\tag{3.7}
$$

这里的近似不是严格相等，而是利用了高噪声条件下数据本身的变化相对较小这一事实。

因此，生成过程可以理解为从高噪声分布出发，逐渐降低噪声水平，最终接近真实数据分布。

### 3.2 从边缘分布推导 Probability Flow ODE

这一部分是理解 EDM 的数学基础。

我们已经知道每个噪声水平对应的边缘分布：

$$
p_\sigma(x)
$$

现在希望构造一个确定性动力系统：

$$
\boxed{
\frac{dX_t}{dt}=v(X_t,t)
}
\tag{3.8}
$$

使它在任意时刻都满足：

$$
X_t\sim p_{\sigma(t)}
\tag{3.9}
$$

其中：

$$
\sigma=\sigma(t)
$$

表示噪声标准差随时间变化。

#### 第一步：高斯卷积满足热方程

由式（3.5）：

$$
p_\sigma(x)
=
\int
p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)\,dy
$$

对于高斯核：

$$
\frac{\partial}{\partial\sigma}
\mathcal N(x;y,\sigma^2I)
=
\sigma\Delta_x
\mathcal N(x;y,\sigma^2I)
\tag{3.10}
$$

其中：

$$
\Delta_x
=
\sum_{j=1}^d
\frac{\partial^2}{\partial x_j^2}
$$

是拉普拉斯算子。

对式（3.5）求导：

$$
\begin{aligned}
\frac{\partial p_\sigma(x)}{\partial\sigma}
&=
\int
p_{\mathrm{data}}(y)
\frac{\partial}{\partial\sigma}
\mathcal N(x;y,\sigma^2I)\,dy\\
&=
\sigma
\int
p_{\mathrm{data}}(y)
\Delta_x
\mathcal N(x;y,\sigma^2I)\,dy\\
&=
\sigma\Delta_xp_\sigma(x)
\end{aligned}
$$

因此：

$$
\boxed{
\frac{\partial p_\sigma}{\partial\sigma}
=
\sigma\Delta_xp_\sigma
}
\tag{3.11}
$$

进一步根据链式法则：

$$
\frac{\partial p_t}{\partial t}
=
\dot\sigma(t)
\frac{\partial p_\sigma}{\partial\sigma}
$$

得到：

$$
\boxed{
\frac{\partial p_t}{\partial t}
=
\sigma(t)\dot\sigma(t)
\Delta_xp_t
}
\tag{3.12}
$$

其中：

$$
p_t(x)=p_{\sigma(t)}(x)
$$

#### 第二步：确定性 ODE 满足连续性方程

对于动力系统：

$$
\frac{dX_t}{dt}=v(X_t,t)
$$

其概率密度满足：

$$
\boxed{
\frac{\partial p_t}{\partial t}
=
-\nabla_x\cdot(p_tv)
}
\tag{3.13}
$$

这是概率质量守恒的数学表达。

为了让 ODE 产生式（3.12）规定的边缘分布，需要：

$$
-\nabla_x\cdot(p_tv)
=
\sigma\dot\sigma\Delta_xp_t
\tag{3.14}
$$

利用：

$$
p_t\nabla_x\log p_t=\nabla_xp_t
\tag{3.15}
$$

可以选择速度场：

$$
\boxed{
v(x,t)
=
-\sigma(t)\dot\sigma(t)
\nabla_x\log p_t(x)
}
\tag{3.16}
$$

验证：

$$
\begin{aligned}
-\nabla_x\cdot(p_tv)
&=
\nabla_x\cdot
\left(
\sigma\dot\sigma
p_t\nabla_x\log p_t
\right)\\
&=
\sigma\dot\sigma
\nabla_x\cdot(\nabla_xp_t)\\
&=
\sigma\dot\sigma\Delta_xp_t
\end{aligned}
$$

与式（3.12）一致。

因此得到：

$$
\boxed{
\frac{dX_t}{dt}
=
-\sigma(t)\dot\sigma(t)
\nabla_x\log p_t(X_t)
}
\tag{3.17}
$$

这就是与该高斯边缘分布演化对应的 Probability Flow ODE。

**注意：** 连续性方程本身并不能唯一确定速度场。这里选择的是与相应扩散过程匹配的概率流速度场，而不是所有可能速度场中唯一的一个。

### 3.3 为什么可以使用 Denoiser 表达 Score？

定义真实 Score：

$$
\boxed{
s^*(x;\sigma)
=
\nabla_x\log p_\sigma(x)
}
\tag{3.18}
$$

EDM 并不要求神经网络直接输出 Score，而是训练一个 Denoiser：

$$
D_\theta(x;\sigma)
$$

其监督目标为：

$$
\boxed{
\mathcal L_{\mathrm{denoise}}
=
\mathbb E_{Y,Z}
\left[
\left\|
D_\theta(Y+\sigma Z;\sigma)-Y
\right\|^2
\right]
}
\tag{3.19}
$$

对于固定的 $\sigma$，平方损失下的最优预测是：

$$
\boxed{
D^*(x;\sigma)
=
\mathbb E[Y\mid X_\sigma=x]
}
\tag{3.20}
$$

这不是一个人为指定的网络结构，而是条件均方误差最小化的数学结论。

#### 第一步：对加噪边缘密度求梯度

根据：

$$
p_\sigma(x)
=
\int p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)\,dy
$$

求梯度：

$$
\nabla_xp_\sigma(x)
=
\int
p_{\mathrm{data}}(y)
\nabla_x\mathcal N(x;y,\sigma^2I)\,dy
$$

由于：

$$
\nabla_x\mathcal N(x;y,\sigma^2I)
=
-\frac{x-y}{\sigma^2}
\mathcal N(x;y,\sigma^2I)
\tag{3.21}
$$

因此：

$$
\nabla_xp_\sigma(x)
=
\frac1{\sigma^2}
\int
(y-x)
p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)\,dy
\tag{3.22}
$$

#### 第二步：将梯度转换为 Score

两边除以 $p_\sigma(x)$：

$$
\begin{aligned}
\nabla_x\log p_\sigma(x)
&=
\frac{\nabla_xp_\sigma(x)}{p_\sigma(x)}\\
&=
\frac1{\sigma^2}
\left[
\frac{
\int y\,p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)\,dy
}{
p_\sigma(x)
}
-x
\right]
\end{aligned}
$$

根据贝叶斯公式：

$$
p(y\mid x;\sigma)
=
\frac{
p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)
}{
p_\sigma(x)
}
\tag{3.23}
$$

因此：

$$
\frac{
\int y\,p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)\,dy
}{
p_\sigma(x)
}
=
\mathbb E[Y\mid X_\sigma=x]
$$

得到：

$$
\boxed{
\nabla_x\log p_\sigma(x)
=
\frac{
D^*(x;\sigma)-x
}{
\sigma^2
}
}
\tag{3.24}
$$

这就是高斯加噪条件下的 Tweedie's Formula。

#### 第三步：代入 Probability Flow ODE

将式（3.24）代入式（3.17）：

$$
\frac{dx}{dt}
=
-\sigma\dot\sigma
\frac{D^*(x;\sigma)-x}{\sigma^2}
$$

整理：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot\sigma(t)}{\sigma(t)}
\left[
x-D^*(x;\sigma(t))
\right]
}
\tag{3.25}
$$

实际使用神经网络近似：

$$
D_\theta(x;\sigma)\approx D^*(x;\sigma)
$$

于是：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot\sigma(t)}{\sigma(t)}
\left[
x-D_\theta(x;\sigma(t))
\right]
}
\tag{3.26}
$$

这一步建立了：

$$
\boxed{
\text{Denoising MSE}
\longrightarrow
\text{Optimal Denoiser}
\longrightarrow
\text{Score}
\longrightarrow
\text{Probability Flow ODE}
}
$$

之间的联系。

### 3.4 为什么 Denoiser 的最优解是条件期望？

这一点对理解后面的网络预条件化非常重要。

设：

$$
X=Y+\sigma Z
$$

我们希望找到一个函数 $D(X)$，最小化：

$$
\mathbb E\|D(X)-Y\|^2
\tag{3.27}
$$

根据全期望公式：

$$
\mathbb E\|D(X)-Y\|^2
=
\mathbb E_X
\left[
\mathbb E[
\|D(X)-Y\|^2
\mid X]
\right]
$$

对于给定的 $X=x$，令：

$$
a=D(x)
$$

需要最小化：

$$
\mathbb E[
\|a-Y\|^2
\mid X=x]
$$

展开：

$$
\begin{aligned}
\mathbb E[
\|a-Y\|^2
\mid X=x]
&=
\|a\|^2
-2a^\top\mathbb E[Y\mid X=x]\\
&\quad+
\mathbb E[\|Y\|^2\mid X=x]
\end{aligned}
$$

对 $a$ 求导：

$$
2a-2\mathbb E[Y\mid X=x]=0
$$

得到：

$$
\boxed{
a^*=\mathbb E[Y\mid X=x]
}
\tag{3.28}
$$

因此：

$$
\boxed{
D^*(x;\sigma)
=
\mathbb E[Y\mid X_\sigma=x]
}
$$

需要特别强调：

**训练标签是真实数据 $Y$，但最优预测一般不是某个唯一的真实样本，而是给定带噪输入后的后验条件均值。**

这也意味着：在多模态数据分布中，高噪声状态对应的最优 Denoiser 可能输出多个模式之间的条件平均。

这并不等于最终生成结果只能是均值，因为扩散采样会沿整个随噪声变化的速度场积分，而不是直接把初始高斯噪声送入一次 Denoiser 后结束。

### 3.5 引入状态缩放 $s(t)$

前面的模型使用：

$$
X_t=Y+\sigma(t)Z
$$

EDM 进一步允许对整个状态进行缩放。

定义未缩放状态：

$$
U_t=Y+\sigma(t)Z
\tag{3.29}
$$

定义缩放后状态：

$$
\boxed{
X_t=s(t)U_t
}
\tag{3.30}
$$

因此：

$$
\boxed{
X_t=s(t)Y+s(t)\sigma(t)Z
}
\tag{3.31}
$$

这里需要区分：

$$
\sigma(t)
$$

是未缩放坐标下的噪声标准差，而：

$$
s(t)\sigma(t)
$$

是缩放后状态中的噪声标准差。

假设 $s(t)>0$。

#### 第一步：对缩放关系求导

$$
X_t=s(t)U_t
$$

所以：

$$
\frac{dX_t}{dt}
=
\dot s(t)U_t
+
s(t)\frac{dU_t}{dt}
$$

代入：

$$
U_t=\frac{X_t}{s(t)}
$$

得到：

$$
\boxed{
\frac{dX_t}{dt}
=
\frac{\dot s(t)}{s(t)}X_t
+
s(t)\frac{dU_t}{dt}
}
\tag{3.32}
$$

#### 第二步：代入未缩放 Probability Flow ODE

未缩放状态满足：

$$
\frac{dU_t}{dt}
=
-\sigma\dot\sigma
\nabla_u\log p_\sigma(U_t)
\tag{3.33}
$$

因此：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
s\sigma\dot\sigma
\nabla_u\log p_\sigma(u)
\bigg|_{u=x/s}
}
\tag{3.34}
$$

这里的 Score：

$$
\nabla_u\log p_\sigma(u)
$$

是对**未缩放坐标 $u$** 求梯度。

这时前面的系数是：

$$
s\sigma\dot\sigma
$$

而不是：

$$
s^2\sigma\dot\sigma
$$

#### 第三步：改写为缩放后坐标的 Score

定义缩放后状态的密度：

$$
q_t(x)
=
s(t)^{-d}
p_{\sigma(t)}
\left(
\frac{x}{s(t)}
\right)
\tag{3.35}
$$

因此：

$$
\log q_t(x)
=
-d\log s(t)
+
\log p_{\sigma(t)}
\left(
\frac{x}{s(t)}
\right)
$$

对 $x$ 求梯度：

$$
\boxed{
\nabla_x\log q_t(x)
=
\frac1{s(t)}
\nabla_u\log p_{\sigma(t)}(u)
\bigg|_{u=x/s(t)}
}
\tag{3.36}
$$

即：

$$
\nabla_u\log p_\sigma(u)
=
s(t)\nabla_x\log q_t(x)
$$

代入式（3.34）：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
s^2\sigma\dot\sigma
\nabla_x\log q_t(x)
}
\tag{3.37}
$$

**式（3.34）和式（3.37）完全等价，但使用了不同坐标系下的 Score。**

这是原笔记中最需要明确区分的地方。

#### 第四步：转换为 Denoiser 形式

根据：

$$
\nabla_u\log p_\sigma(u)
=
\frac{D^*(u;\sigma)-u}{\sigma^2}
$$

代入式（3.34）：

$$
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
s\frac{\dot\sigma}{\sigma}
\left[
D^*\left(\frac{x}{s};\sigma\right)
-\frac{x}{s}
\right]
$$

展开：

$$
\boxed{
\frac{dx}{dt}
=
\left(
\frac{\dot s}{s}
+
\frac{\dot\sigma}{\sigma}
\right)x
-
s\frac{\dot\sigma}{\sigma}
D^*
\left(
\frac{x}{s};\sigma
\right)
}
\tag{3.38}
$$

实际采样时使用：

$$
D_\theta\approx D^*
$$

这就是 EDM 通用采样框架中的 ODE 速度表达式。

### 3.6 VP、VE 与 EDM 的统一

现在使用：

$$
\boxed{
X_t=s(t)Y+s(t)\sigma(t)Z
}
\tag{3.39}
$$

来表示不同扩散过程。

#### 3.6.1 VE

VE 的边缘加噪形式：

$$
X_t=Y+\sigma(t)Z
$$

因此：

$$
\boxed{
s(t)=1
}
\tag{3.40}
$$

但 VE 并不要求：

$$
\sigma(t)=t
$$

例如，经典 VE SDE 可以采用指数噪声调度：

$$
\sigma(t)
=
\sigma_{\min}
\left(
\frac{\sigma_{\max}}{\sigma_{\min}}
\right)^t
\tag{3.41}
$$

其特点是状态不缩放，但噪声标准差随时间变化。

#### 3.6.2 VP

VP 的边缘分布通常写为：

$$
X_t=\alpha(t)Y+\bar\sigma(t)Z
\tag{3.42}
$$

其中：

$$
\alpha(t)^2+\bar\sigma(t)^2=1
$$

对于标准 VP SDE：

$$
dX_t
=
-\frac12\beta(t)X_t\,dt
+
\sqrt{\beta(t)}\,dW_t
\tag{3.43}
$$

有：

$$
\alpha(t)
=
\exp
\left(
-\frac12\int_0^t\beta(u)\,du
\right)
\tag{3.44}
$$

以及：

$$
\bar\sigma(t)
=
\sqrt{1-\alpha(t)^2}
\tag{3.45}
$$

与统一形式比较：

$$
s(t)Y+s(t)\sigma(t)Z
=
\alpha(t)Y+\bar\sigma(t)Z
$$

得到：

$$
\boxed{
s(t)=\alpha(t)
}
\tag{3.46}
$$

$$
\boxed{
\sigma(t)
=
\frac{\bar\sigma(t)}{\alpha(t)}
}
\tag{3.47}
$$

因此 VP 与 VE 可以通过不同的 $s(t)$ 和 $\sigma(t)$ 统一表示。

#### 3.6.3 EDM 推荐形式

EDM 推荐：

$$
\boxed{
s(t)=1,\qquad \sigma(t)=t
}
\tag{3.48}
$$

此时：

$$
X_t=Y+tZ
$$

对应的 Probability Flow ODE：

$$
\frac{dx}{dt}
=
-\sigma\dot\sigma\nabla_x\log p_\sigma(x)
$$

因为：

$$
\sigma=t,\qquad\dot\sigma=1
$$

所以：

$$
\frac{dx}{dt}
=
-t\nabla_x\log p_t(x)
$$

再使用 Tweedie's Formula：

$$
\nabla_x\log p_t(x)
=
\frac{D^*(x;t)-x}{t^2}
$$

得到：

$$
\boxed{
\frac{dx}{dt}
=
\frac{x-D^*(x;t)}{t}
}
\tag{3.49}
$$

使用神经网络近似后：

$$
\boxed{
\frac{dx}{dt}
=
\frac{x-D_\theta(x;t)}{t}
}
\tag{3.50}
$$

这个形式在边缘加噪方式上属于 VE 类型，但并不意味着 EDM 完全采用原始 VE 模型的训练和采样设计。

**Section 2 的核心结论：EDM 通过统一的边缘分布与 ODE 表达式，将不同扩散模型的设计拆分成可以独立研究的部分。**

---




EDM 推荐的 Probability Flow ODE 为：

$$
\boxed{
\frac{dx}{dt}
=
\frac{x-D_\theta(x;t)}{t}
}
\tag{4.1}
$$

其中：

$$
\boxed{\sigma(t)=t}
$$

所以在推荐参数化下，**时间变量 $t$ 的数值就是噪声标准差**。

---

# 四、Section 3：Deterministic Sampling

## 4.1 确定性采样究竟在求解什么？

在上一章，我们从高斯加噪分布推导出了：

$$
\frac{dx}{dt}
=
-\sigma(t)\dot\sigma(t)
\nabla_x\log p_{\sigma(t)}(x)
\tag{4.2}
$$

通过 Tweedie's Formula：

$$
\nabla_x\log p_\sigma(x)
=
\frac{D^*(x;\sigma)-x}{\sigma^2}
$$

得到：

$$
\frac{dx}{dt}
=
\frac{\dot\sigma(t)}{\sigma(t)}
\left[x-D^*(x;\sigma(t))\right]
\tag{4.3}
$$

当：

$$
\sigma(t)=t
$$

时：

$$
\boxed{
\frac{dx}{dt}
=
v_\theta(x,t)
=
\frac{x-D_\theta(x;t)}{t}
}
\tag{4.4}
$$

这里的速度场：

$$
v_\theta(x,t)
$$

是由神经网络定义的。

给定高噪声初始状态：

$$
x(t_{\max})\sim
\mathcal N(0,t_{\max}^2I)
$$

我们希望从：

$$
t_{\max}
$$

逐渐积分到：

$$
t=0
$$

最终得到：

$$
x(0)\sim p_{\mathrm{data}}
$$

理想情况下，这个过程能够将高斯分布输运到真实数据分布。

需要注意两个问题。

**第一，ODE 的时间方向是递减的。**

因为生成过程是去噪：

$$
t_{\max}\rightarrow0
$$

所以数值积分的时间增量：

$$
h_i=t_{i+1}-t_i
$$

通常是负数。

**第二，ODE 的真实速度场并不知道。**

实际使用的是：

$$
D_\theta\approx D^*
$$

因此存在两种不同的误差：

$$
\boxed{
\text{模型误差}
+
\text{数值积分误差}
}
\tag{4.5}
$$

其中：

- 模型误差来自 $D_\theta\neq D^*$。
- 数值积分误差来自有限步长的 ODE 求解。

EDM 的确定性采样部分主要研究如何减小第二类误差。

---

## 4.2 Euler：一阶 ODE 求解器

考虑一般 ODE：

$$
\frac{dx}{dt}=v(x,t)
\tag{4.6}
$$

在当前状态：

$$
(x_i,t_i)
$$

附近，对解进行 Taylor 展开：

$$
x(t_i+h)
=
x(t_i)
+
h\frac{dx}{dt}\bigg|_{t_i}
+
\frac{h^2}{2}
\frac{d^2x}{dt^2}\bigg|_{t_i}
+
O(h^3)
\tag{4.7}
$$

由于：

$$
\frac{dx}{dt}=v(x,t)
$$

保留一阶项：

$$
\boxed{
x_{i+1}
=
x_i+h_iv(x_i,t_i)
}
\tag{4.8}
$$

其中：

$$
h_i=t_{i+1}-t_i
$$

这就是 Euler 方法。

对于 EDM：

$$
v_\theta(x_i,t_i)
=
\frac{x_i-D_\theta(x_i;t_i)}{t_i}
$$

因此：

$$
\boxed{
x_{i+1}
=
x_i+
(t_{i+1}-t_i)
\frac{x_i-D_\theta(x_i;t_i)}{t_i}
}
\tag{4.9}
$$

每一步只需要调用一次神经网络：

$$
\boxed{\mathrm{NFE}=1}
$$

这里 NFE 表示 Network Function Evaluations，即网络前向计算次数。

### Euler 的误差从哪里来？

Euler 假设当前时间区间内的速度近似保持不变：

$$
v(x(t),t)\approx v(x_i,t_i)
$$

但真实速度会随着状态和时间变化。

由链式法则：

$$
\frac{d^2x}{dt^2}
=
\frac{d}{dt}v(x(t),t)
$$

得到：

$$
\boxed{
\frac{d^2x}{dt^2}
=
\frac{\partial v}{\partial t}
+
\frac{\partial v}{\partial x}v
}
\tag{4.10}
$$

因此 Euler 忽略的主要误差项是：

$$
\frac{h_i^2}{2}
\left(
\frac{\partial v}{\partial t}
+
\frac{\partial v}{\partial x}v
\right)
$$

局部截断误差：

$$
\boxed{
O(h_i^2)
}
\tag{4.11}
$$

在常规光滑性与稳定性条件下，全局误差为：

$$
\boxed{
O(h)
}
\tag{4.12}
$$

其中 $h$ 表示最大步长。

这说明，当 ODE 轨迹弯曲较明显，或者速度场变化较快时，Euler 需要更小的步长才能保持精度。

---

## 4.3 Heun：二阶 ODE 求解器

EDM 推荐使用二阶 Heun 方法。

其基本思想是：

> 不只使用积分区间起点的速度，还估计终点的速度，然后用两者的平均值更新状态。

考虑：

$$
\frac{dx}{dt}=v(x,t)
$$

### 第一步：计算起点速度

$$
\boxed{
d_i=v(x_i,t_i)
}
\tag{4.13}
$$

对于 EDM：

$$
d_i
=
\frac{x_i-D_\theta(x_i;t_i)}{t_i}
$$

### 第二步：Euler 预测终点

$$
\boxed{
\tilde x_{i+1}
=
x_i+h_id_i
}
\tag{4.14}
$$

这只是一个临时预测值。

### 第三步：计算预测终点的速度

$$
\boxed{
\tilde d_{i+1}
=
v(\tilde x_{i+1},t_{i+1})
}
\tag{4.15}
$$

对于 EDM：

$$
\tilde d_{i+1}
=
\frac{
\tilde x_{i+1}
-
D_\theta(\tilde x_{i+1};t_{i+1})
}{
t_{i+1}
}
$$

### 第四步：使用平均速度更新

$$
\boxed{
x_{i+1}
=
x_i+
\frac{h_i}{2}
(d_i+\tilde d_{i+1})
}
\tag{4.16}
$$

因此，Heun 的完整形式为：

$$
\boxed{
\begin{aligned}
d_i&=v(x_i,t_i)\\
\tilde x_{i+1}&=x_i+h_id_i\\
\tilde d_{i+1}&=v(\tilde x_{i+1},t_{i+1})\\
x_{i+1}&=
x_i+\frac{h_i}{2}(d_i+\tilde d_{i+1})
\end{aligned}
}
\tag{4.17}
$$

### 为什么 Heun 是二阶方法？

从积分形式出发：

$$
x(t_{i+1})
=
x(t_i)+
\int_{t_i}^{t_{i+1}}v(x(t),t)\,dt
\tag{4.18}
$$

Euler 使用左端点近似：

$$
\int_{t_i}^{t_{i+1}}v\,dt
\approx h_iv(x_i,t_i)
$$

Heun 则使用梯形公式：

$$
\int_{t_i}^{t_{i+1}}v\,dt
\approx
\frac{h_i}{2}
\left[
v(x_i,t_i)+v(x_{i+1},t_{i+1})
\right]
\tag{4.19}
$$

由于真实的：

$$
x_{i+1}
$$

尚未得到，因此先使用 Euler 预测：

$$
\tilde x_{i+1}
$$

代替它。

这样可以消除 Euler 主要的二阶局部误差项。

因此：

$$
\boxed{
\text{Heun 局部误差}=O(h_i^3)
}
\tag{4.20}
$$

全局误差通常为：

$$
\boxed{
O(h^2)
}
\tag{4.21}
$$

### Heun 为什么需要两次网络推理？

因为需要计算两个不同位置的速度：

$$
v(x_i,t_i)
$$

以及：

$$
v(\tilde x_{i+1},t_{i+1})
$$

而每个速度都需要调用 Denoiser。

所以通常：

$$
\boxed{
\mathrm{NFE}=2\text{ / step}
}
$$

不过 EDM 在最后一步采用 Euler，因此如果总共有 $N$ 个积分区间，通常：

$$
\boxed{
\mathrm{NFE}=2N-1
}
\tag{4.22}
$$

这里假设没有额外的网络计算。

---

## 4.4 为什么最后一步使用 Euler？

EDM 最后一个噪声水平为：

$$
t_N=0
$$

但速度场表达式为：

$$
v_\theta(x,t)
=
\frac{x-D_\theta(x;t)}{t}
$$

当：

$$
t=0
$$

时，公式中出现除零问题。

虽然理想情况下：

$$
D^*(x;t)\rightarrow x
$$

但直接计算：

$$
\frac{x-D_\theta(x;t)}{t}
$$

在数值上仍可能不稳定。

因此 EDM 最后一步不再计算终点速度，而是使用 Euler：

$$
\boxed{
x_N
=
x_{N-1}
-
t_{N-1}
v_\theta(x_{N-1},t_{N-1})
}
\tag{4.23}
$$

代入：

$$
v_\theta(x,t)
=
\frac{x-D_\theta(x;t)}{t}
$$

得到：

$$
\begin{aligned}
x_N
&=
x_{N-1}
-
t_{N-1}
\frac{
x_{N-1}-D_\theta(x_{N-1};t_{N-1})
}{
t_{N-1}
}\\
&=
D_\theta(x_{N-1};t_{N-1})
\end{aligned}
$$

因此：

$$
\boxed{
x_N=D_\theta(x_{N-1};t_{N-1})
}
\tag{4.24}
$$

这说明最后一步 Euler 实际上等价于直接输出当前 Denoiser 的预测结果。

---

## 4.5 为什么 EDM 不使用均匀时间步长？

这是论文采样设计中的另一个重要问题。

假设：

$$
t_{\max}=80,\qquad t_{\min}=0.002
$$

如果使用均匀网格：

$$
t_i
=
t_{\max}
+
\frac{i}{N-1}(t_{\min}-t_{\max})
\tag{4.25}
$$

那么：

$$
h_i=t_{i+1}-t_i
$$

基本保持不变。

但不同噪声区域的 ODE 并不具有相同的数值性质。

### 高噪声区域

当：

$$
t\gg\sigma_{\mathrm{data}}
$$

时：

$$
X_t=Y+tZ
$$

主要由高斯噪声决定。

如果数据均值为零，则：

$$
D^*(x;t)
$$

通常只包含相对较弱的数据结构信息。

速度场：

$$
v^*(x,t)
=
\frac{x-D^*(x;t)}{t}
$$

往往比较接近：

$$
\frac{x}{t}
$$

在高噪声区域，生成轨迹可能相对平滑。

### 低噪声区域

当：

$$
t\rightarrow0
$$

时：

$$
X_t\approx Y
$$

Denoiser 开始恢复更细致的数据结构。

此时速度场可能随状态和噪声水平发生较快变化。

因此，为了控制数值积分误差，通常需要在低噪声区域使用更细的时间网格。

但需要强调：

**并不是所有数据分布下，低噪声区域的轨迹曲率都一定更大。**

这是一种有实验支持的采样设计经验，而不是普遍成立的数学定理。

---

## 4.6 EDM 的非均匀噪声网格

论文采用：

$$
\boxed{
t_i
=
\left[
t_{\max}^{1/\rho}
+
\frac{i}{N-1}
\left(
t_{\min}^{1/\rho}
-
t_{\max}^{1/\rho}
\right)
\right]^\rho
}
\tag{4.26}
$$

其中：

$$
i=0,1,\ldots,N-1
$$

并额外设置：

$$
\boxed{t_N=0}
\tag{4.27}
$$

这里：

- $N$：采样积分区间数。
- $t_{\max}$：初始噪声标准差。
- $t_{\min}$：最后一个非零噪声标准差。
- $\rho$：控制网格密度分配的参数。

EDM 推荐：

$$
\boxed{\rho=7}
\tag{4.28}
$$

### 为什么这个公式能够产生非均匀步长？

定义一个新的变量：

$$
\boxed{
r=t^{1/\rho}
}
\tag{4.29}
$$

则：

$$
t=r^\rho
$$

EDM 实际上是在 $r$ 空间使用均匀网格：

$$
r_i
=
r_{\max}
+
\frac{i}{N-1}(r_{\min}-r_{\max})
\tag{4.30}
$$

再映射回：

$$
t_i=r_i^\rho
$$

对 $r$ 求导：

$$
\frac{dt}{dr}
=
\rho r^{\rho-1}
\tag{4.31}
$$

因为：

$$
r=t^{1/\rho}
$$

所以：

$$
\boxed{
\frac{dt}{dr}
=
\rho t^{1-1/\rho}
}
\tag{4.32}
$$

对于固定的：

$$
\Delta r
$$

有近似关系：

$$
\boxed{
|\Delta t|
\approx
\rho t^{1-1/\rho}|\Delta r|
}
\tag{4.33}
$$

当：

$$
\rho>1
$$

时：

$$
1-\frac1\rho>0
$$

因此：

- $t$ 较大时，$|\Delta t|$ 较大。
- $t$ 较小时，$|\Delta t|$ 较小。

所以：

$$
\boxed{
\text{高噪声区域大步长，低噪声区域小步长}
}
\tag{4.34}
$$

### 为什么设计的是噪声 $\sigma_i$，而不是直接设计 $dt_i$？

因为在 EDM 推荐的参数化下：

$$
\boxed{t=\sigma}
$$

因此：

$$
\boxed{
dt_i=t_{i+1}-t_i
=
\sigma_{i+1}-\sigma_i
}
\tag{4.35}
$$

设计离散噪声水平：

$$
\sigma_0,\sigma_1,\ldots,\sigma_N
$$

实际上就已经确定了积分步长：

$$
\Delta t_i=\sigma_{i+1}-\sigma_i
$$

两者不是两个独立问题。

之所以用噪声标准差表示，是因为扩散模型的分布、Denoiser 和速度场都直接依赖于 $\sigma$。

因此：

$$
\boxed{
\text{噪声网格设计}
=
\text{ODE 时间网格设计}
}
$$

但这个等式依赖于：

$$
\sigma(t)=t
$$

如果使用其他噪声调度，则：

$$
\Delta\sigma_i
\neq
\Delta t_i
$$

一般不再相等。

---

## 4.7 时间参数化与 ODE 轨迹曲率

这一部分是对原论文数值分析思想的进一步解释。

考虑同一条 ODE 解轨迹：

$$
x=x(t)
$$

引入新的时间变量：

$$
r=g(t)
$$

并假设 $g$ 单调可微。

根据链式法则：

$$
\frac{dx}{dr}
=
\frac{dx}{dt}
\frac{dt}{dr}
\tag{4.36}
$$

如果：

$$
\frac{dx}{dt}=v(x,t)
$$

那么：

$$
\boxed{
\frac{dx}{dr}
=
v(x,t(r))
\frac{dt}{dr}
}
\tag{4.37}
$$

这说明：

**时间参数化可以改变同一条轨迹在不同时间位置上的运动速度。**

继续求导：

$$
\frac{d^2x}{dr^2}
=
\frac{d}{dr}
\left(
\frac{dx}{dt}\frac{dt}{dr}
\right)
$$

得到：

$$
\boxed{
\frac{d^2x}{dr^2}
=
\frac{d^2x}{dt^2}
\left(\frac{dt}{dr}\right)^2
+
\frac{dx}{dt}
\frac{d^2t}{dr^2}
}
\tag{4.38}
$$

因此即使空间中的几何轨迹不变，不同时间参数化下：

$$
\frac{d^2x}{dr^2}
$$

也会改变。

而 Euler、Heun 等数值方法的局部误差与这些导数有关。

所以：

$$
\boxed{
\text{同一条生成轨迹，在不同时间参数化下，数值求解难度可能不同}
}
$$

这解释了为什么 EDM 不仅研究积分器，也研究噪声调度和时间参数化。

不过，不能简单地认为某一种参数化能够对所有模型和数据分布实现最小数值误差。

---

## 4.8 为什么 EDM 选择 Heun，而不是更高阶的 Runge–Kutta？

从纯数值分析角度看，更高阶的求解器通常能够在足够光滑的条件下取得更高的渐近精度。

例如：

| 方法 | 阶数 | 典型每步 NFE |
|---|---:|---:|
| Euler | 1 | 1 |
| Heun | 2 | 2 |
| RK4 | 4 | 4 |

但扩散采样不是求解一个已知精确速度场。

实际使用的是：

$$
v_\theta(x,t)
=
v^*(x,t)+e_\theta(x,t)
\tag{4.39}
$$

其中：

$$
e_\theta
$$

表示网络预测误差。

因此，总体误差至少包含：

$$
\boxed{
E_{\mathrm{total}}
\approx
E_{\mathrm{model}}
+
E_{\mathrm{solver}}
}
\tag{4.40}
$$

这里的加法是误差来源的概念性分解，不表示所有误差度量下都严格可加。

如果网络误差已经占据主要部分，那么进一步提高求解器阶数，可能只能获得有限收益。

此外，更高阶求解器每一步需要更多网络调用。

EDM 的实验表明，二阶 Heun 在采样质量和 NFE 之间具有较好的权衡。

因此：

$$
\boxed{
\text{Heun 是经验上有效的工程选择，而不是理论上唯一最优的 Solver}
}
$$

---

## 4.9 Section 3 总结

确定性采样部分可以归纳为：

$$
\boxed{
\text{EDM PF ODE}
\rightarrow
\text{Heun}
\rightarrow
\text{非均匀噪声网格}
}
$$

其中：

$$
\boxed{
v_\theta(x,t)
=
\frac{x-D_\theta(x;t)}{t}
}
$$

$$
\boxed{
x_{i+1}
=
x_i+
\frac{t_{i+1}-t_i}{2}
(d_i+\tilde d_{i+1})
}
$$

$$
\boxed{
t_i
=
\left[
t_{\max}^{1/\rho}
+
\frac{i}{N-1}
\left(
t_{\min}^{1/\rho}-t_{\max}^{1/\rho}
\right)
\right]^\rho
}
$$

核心思想是：

**在有限 NFE 预算下，使用更合理的积分方法和时间网格，提高对模型定义的 PF ODE 的求解精度。**

---

# 五、Section 4：Stochastic Sampling

## 5.1 为什么已经有 PF ODE，还要研究随机采样？

对于理想扩散模型：

$$
D_\theta=D^*
$$

如果数值积分也完全精确，那么确定性 PF ODE 可以正确输运目标边缘分布。

但实际存在：

$$
D_\theta\neq D^*
$$

以及有限步数的数值误差。

因此生成过程中实际状态分布：

$$
q_t(x)
$$

可能偏离理想分布：

$$
p_t(x)
$$

即：

$$
\boxed{
q_t(x)\neq p_t(x)
}
\tag{5.1}
$$

随机采样的一个研究动机是：

> 通过在生成过程中引入适当随机性，是否能够改善这种分布偏差？

EDM 发现，在部分模型和数据集上，适量随机性确实能够改善 FID。

但对于某些训练质量更好的模型，确定性采样反而更好。

因此论文没有将随机性视为必要条件，而是将其作为独立的采样设计变量。

---

## 5.2 PF ODE 与反向 SDE 的关系

先考虑 VE 型正向扩散：

$$
\boxed{
dX_t
=
g(t)dW_t
}
\tag{5.2}
$$

其边缘分布满足：

$$
X_t\overset d=
Y+\sigma(t)Z
$$

当：

$$
\sigma(0)=0
$$

时，需要：

$$
\boxed{
g(t)^2
=
\frac{d\sigma(t)^2}{dt}
=
2\sigma(t)\dot\sigma(t)
}
\tag{5.3}
$$

因此：

$$
\boxed{
dX_t
=
\sqrt{2\sigma(t)\dot\sigma(t)}\,dW_t
}
\tag{5.4}
$$

这里要求正向时间中：

$$
\sigma(t)\dot\sigma(t)\geq0
$$

对应的 PF ODE：

$$
\boxed{
dX_t
=
-\sigma(t)\dot\sigma(t)
\nabla_x\log p_t(X_t)\,dt
}
\tag{5.5}
$$

反向时间 SDE 则包含 Score 漂移与布朗运动噪声。

在沿原始时间变量反向积分的记号下，形式为：

$$
dX_t
=
-2\sigma(t)\dot\sigma(t)
\nabla_x\log p_t(X_t)\,dt
+
g(t)\,d\bar W_t
\tag{5.6}
$$

其中积分方向是：

$$
t_{\max}\rightarrow0
$$

并且 $d\bar W_t$ 表示反向时间布朗运动增量。

为了避免时间符号混淆，也可以引入一个单调增加的反向采样时间：

$$
\tau=T-t
$$

然后重新写出相应 SDE。

关键是：

$$
\boxed{
\text{PF ODE 与反向 SDE 可以具有相同的理想边缘分布}
}
$$

但它们的样本路径不同。

PF ODE 的路径由初始状态确定，而反向 SDE 在生成过程中持续引入随机性。

---

## 5.3 Langevin 动力学为什么能够保持目标分布？

为了理解 EDM 的 Stochastic Churn，需要先理解 Langevin 动力学。

假设当前固定噪声水平为：

$$
\sigma
$$

对应目标分布：

$$
p_\sigma(x)
$$

考虑 SDE：

$$
\boxed{
dX_\tau
=
a\nabla_x\log p_\sigma(X_\tau)\,d\tau
+
\sqrt{2a}\,dW_\tau
}
\tag{5.7}
$$

其中：

$$
a>0
$$

是扩散强度。

这个 SDE 的 Fokker–Planck 方程为：

$$
\frac{\partial q}{\partial\tau}
=
-a\nabla\cdot
\left(
q\nabla\log p_\sigma
\right)
+
a\Delta q
\tag{5.8}
$$

如果：

$$
q=p_\sigma
$$

则：

$$
p_\sigma\nabla\log p_\sigma
=
\nabla p_\sigma
$$

所以：

$$
\begin{aligned}
\frac{\partial q}{\partial\tau}
&=
-a\nabla\cdot(\nabla p_\sigma)
+a\Delta p_\sigma\\
&=0
\end{aligned}
$$

因此：

$$
\boxed{
p_\sigma
\text{ 是 Langevin 动力学的不变分布}
}
\tag{5.9}
$$

这意味着，如果当前样本已经服从：

$$
p_\sigma
$$

那么理想 Langevin 动力学不会改变其边缘分布。

如果当前分布偏离：

$$
p_\sigma
$$

则在适当条件下，Langevin 动力学可能使分布逐渐接近目标分布。

这为在扩散采样中加入随机性提供了数学动机。

但需要注意：

**EDM 的 Stochastic Churn 并不是直接在每个噪声水平运行完整的 Langevin MCMC。**

它采用的是一种更简单的分裂式采样方法。

---

## 5.4 EDM 的随机采样思想

EDM 采用：

$$
\boxed{
\text{提高噪声水平}
\rightarrow
\text{添加高斯噪声}
\rightarrow
\text{使用 Heun 去噪}
}
\tag{5.10}
$$

假设当前状态为：

$$
(x_i,t_i)
$$

其中：

$$
t_i=\sigma_i
$$

### 第一步：确定临时噪声水平

定义：

$$
\boxed{
\hat t_i=(1+\gamma_i)t_i
}
\tag{5.11}
$$

其中：

$$
\gamma_i\geq0
$$

所以：

$$
\hat t_i\geq t_i
$$

这意味着先将当前状态移动到一个更高的噪声水平。

### 第二步：添加高斯噪声

采样：

$$
\epsilon_i\sim
\mathcal N(0,S_{\mathrm{noise}}^2I)
$$

然后：

$$
\boxed{
\hat x_i
=
x_i+
\sqrt{\hat t_i^2-t_i^2}\,\epsilon_i
}
\tag{5.12}
$$

为什么是：

$$
\sqrt{\hat t_i^2-t_i^2}
$$

而不是：

$$
\hat t_i-t_i
$$

因为独立高斯噪声的方差相加。

假设：

$$
x_i=Y+t_iZ_1
$$

其中：

$$
Z_1\sim\mathcal N(0,I)
$$

再添加：

$$
\eta Z_2
$$

其中：

$$
Z_2\sim\mathcal N(0,I)
$$

且：

$$
Z_1\perp Z_2
$$

则：

$$
t_iZ_1+\eta Z_2
\sim
\mathcal N(0,(t_i^2+\eta^2)I)
$$

希望总噪声标准差为：

$$
\hat t_i
$$

因此：

$$
t_i^2+\eta^2=\hat t_i^2
$$

得到：

$$
\boxed{
\eta=\sqrt{\hat t_i^2-t_i^2}
}
\tag{5.13}
$$

当：

$$
S_{\mathrm{noise}}=1
$$

且：

$$
x_i\sim p(x;t_i)
$$

时：

$$
\boxed{
\hat x_i\sim p(x;\hat t_i)
}
\tag{5.14}
$$

如果：

$$
S_{\mathrm{noise}}\neq1
$$

则这一步不再严格对应从 $t_i$ 到 $\hat t_i$ 的理想高斯边缘分布变换。

### 第三步：计算新的 ODE 速度

在加噪后的状态：

$$
(\hat x_i,\hat t_i)
$$

计算：

$$
\boxed{
d_i
=
\frac{
\hat x_i-D_\theta(\hat x_i;\hat t_i)
}{
\hat t_i
}
}
\tag{5.15}
$$

注意，这里的 Denoiser 输入不是原来的：

$$
(x_i,t_i)
$$

而是：

$$
(\hat x_i,\hat t_i)
$$

因为随机扰动已经改变了状态和噪声水平。

### 第四步：Euler 预测

从：

$$
\hat t_i
$$

积分到：

$$
t_{i+1}
$$

定义：

$$
h_i=t_{i+1}-\hat t_i
$$

计算：

$$
\boxed{
\tilde x_{i+1}
=
\hat x_i+h_id_i
}
\tag{5.16}
$$

### 第五步：Heun 修正

如果：

$$
t_{i+1}>0
$$

计算：

$$
\tilde d_{i+1}
=
\frac{
\tilde x_{i+1}
-
D_\theta(\tilde x_{i+1};t_{i+1})
}{
t_{i+1}
}
\tag{5.17}
$$

然后：

$$
\boxed{
x_{i+1}
=
\hat x_i+
\frac{h_i}{2}
(d_i+\tilde d_{i+1})
}
\tag{5.18}
$$

如果：

$$
t_{i+1}=0
$$

则使用 Euler。

---

## 5.5 Stochastic Churn 的参数设计

EDM 使用：

$$
\boxed{
\gamma_i=
\begin{cases}
\min\left(
\dfrac{S_{\mathrm{churn}}}{N},
\sqrt2-1
\right),
&
t_i\in[S_{\mathrm{tmin}},S_{\mathrm{tmax}}]
\$$6pt]
0,&\text{其他情况}
\end{cases}
}
\tag{5.19}
$$

其中：

| 参数 | 含义 |
|---|---|
| $S_{\mathrm{churn}}$ | 控制整体随机扰动强度 |
| $S_{\mathrm{tmin}}$ | 允许随机扰动的最低噪声水平 |
| $S_{\mathrm{tmax}}$ | 允许随机扰动的最高噪声水平 |
| $S_{\mathrm{noise}}$ | 调整新增高斯噪声的标准差 |
| $N$ | 积分区间数量 |

为什么限制：

$$
\gamma_i\leq\sqrt2-1
$$

因为：

$$
\hat t_i=(1+\gamma_i)t_i
$$

所以：

$$
\hat t_i\leq\sqrt2t_i
$$

平方：

$$
\hat t_i^2\leq2t_i^2
$$

因此：

$$
\boxed{
\hat t_i^2-t_i^2\leq t_i^2
}
\tag{5.20}
$$

也就是说，单次新增的理想噪声方差不超过当前噪声方差。

---

## 5.6 Stochastic Churn 为什么可能改善采样？

假设当前：

$$
x_i\sim p(x;t_i)
$$

首先执行精确高斯加噪：

$$
x_i\rightarrow\hat x_i
$$

得到：

$$
\hat x_i\sim p(x;\hat t_i)
$$

然后使用精确 PF ODE：

$$
\hat t_i\rightarrow t_{i+1}
$$

得到：

$$
x_{i+1}\sim p(x;t_{i+1})
$$

因此理想情况下：

$$
\boxed{
p_{t_i}
\xrightarrow{\text{加噪}}
p_{\hat t_i}
\xrightarrow{\text{ODE}}
p_{t_{i+1}}
}
\tag{5.21}
$$

这个过程可以保持正确的边缘分布演化。

但实际网络存在误差：

$$
D_\theta\neq D^*
$$

因此：

$$
q_t\neq p_t
$$

适量随机扰动可能使样本重新进入网络能够有效修正的状态区域，从而改善生成分布。

不过这只是一种机制解释，并不保证随机性必然有效。

因为随机扰动同时也可能破坏已经恢复的细节。

因此：

$$
\boxed{
\text{随机扰动可能改善生成质量，也可能降低生成质量}
}
$$

这取决于：

- 模型本身的准确程度。
- 数据分布的复杂程度。
- 噪声注入强度。
- 随机扰动发生的噪声区间。
- 后续 ODE 求解器的精度。

---

## 5.7 为什么不直接使用反向 SDE？

这是理解 EDM 设计动机的重要问题。

从数学角度，完全可以直接使用反向 SDE：

$$
dX_t
=
b_\theta(X_t,t)\,dt
+
g(t)\,d\bar W_t
\tag{5.22}
$$

然后使用 Euler–Maruyama 等随机数值积分方法求解。

但 EDM 希望独立研究随机性对生成质量的影响。

如果直接替换为另一套 SDE 采样器，可能同时改变：

- 漂移项。
- 扩散项。
- 时间网格。
- 数值积分方法。
- 随机扰动强度。

这样就很难判断性能变化来自哪个因素。

而 Stochastic Churn 保留了同一个 Denoiser 和同一个 Heun Solver，只在积分前增加随机扰动。

当：

$$
S_{\mathrm{churn}}=0
$$

时：

$$
\gamma_i=0
$$

所以：

$$
\hat t_i=t_i
$$

$$
\hat x_i=x_i
$$

于是恢复为确定性 Heun。

因此：

$$
\boxed{
\text{Stochastic Churn 是一种便于控制随机性强度的采样设计}
}
$$

但它不是所有反向 SDE 的通用离散化方法。

---

## 5.8 Section 4 总结

EDM 的随机采样并不是简单地：

$$
\text{Heun}+\text{任意高斯噪声}
$$

而是：

$$
\boxed{
\text{在当前噪声水平重新加噪}
+
\text{从更高噪声水平执行 ODE 去噪}
}
$$

其关键公式为：

$$
\boxed{
\hat t_i=(1+\gamma_i)t_i
}
$$

$$
\boxed{
\hat x_i
=
x_i+
\sqrt{\hat t_i^2-t_i^2}\,\epsilon_i
}
$$

$$
\boxed{
x_{i+1}
=
\operatorname{Heun}
(\hat x_i,\hat t_i,t_{i+1})
}
$$

核心结论是：

**随机性不是生成高质量样本的必要条件，而是可以根据模型和数据集进行调节的采样设计变量。**

---

# 六、Section 5：Training——网络预条件化与训练策略

这是 EDM 全文中与模型训练最直接相关的部分，也是最容易产生理解偏差的部分。

本节按照以下逻辑展开：

$$
\boxed{
\text{原始训练目标}
\rightarrow
\text{最优 Denoiser}
\rightarrow
\text{网络重新参数化}
\rightarrow
\text{各系数推导}
\rightarrow
\text{Loss 变换}
\rightarrow
\text{噪声采样分布}
}
$$

## 6.1 首先明确：EDM 究竟希望模型学到什么？

定义真实数据：

$$
Y\sim p_{\mathrm{data}}
$$

独立高斯噪声：

$$
Z\sim\mathcal N(0,I)
$$

带噪输入：

$$
\boxed{X=Y+\sigma Z}
\tag{6.1}
$$

对于给定的噪声水平 $\sigma$，最原始的去噪训练目标为：

$$
\boxed{
\mathcal L_{\mathrm{raw}}(\theta;\sigma)
=
\mathbb E_{Y,Z}
\left[
\|D_\theta(X;\sigma)-Y\|^2
\right]
}
\tag{6.2}
$$

这里有三个需要区分的对象。

**第一，训练标签是 $Y$。**

每个训练样本都具有一个真实的干净数据：

$$
Y
$$

我们希望网络输出尽可能接近它。

**第二，模型实际能够学习的是条件映射。**

因为网络输入只有：

$$
X,\sigma
$$

而不知道：

$$
Y,Z
$$

所以它无法一般性地恢复某个唯一的 $Y$。

平方损失的理论最优解为：

$$
\boxed{
D^*(x;\sigma)
=
\mathbb E[Y\mid X=x]
}
\tag{6.3}
$$

**第三，EDM 最终需要的是 Denoiser，而不一定要求内部网络直接预测 $Y$。**

因为通过 Tweedie's Formula：

$$
\boxed{
\nabla_x\log p_\sigma(x)
=
\frac{D^*(x;\sigma)-x}{\sigma^2}
}
\tag{6.4}
$$

只要能够获得：

$$
D_\theta\approx D^*
$$

就可以构造 Score 和 PF ODE。

因此 EDM 的最终学习目标是：

$$
\boxed{
D_\theta(x;\sigma)
\approx
\mathbb E[Y\mid X=x]
}
\tag{6.5}
$$

这一定义在后面的网络预条件化过程中始终不变。

---

## 6.2 为什么不直接使用一个网络预测 $Y$？

最简单的方案是：

$$
\boxed{
D_\theta(x;\sigma)=F_\theta(x,\sigma)
}
\tag{6.6}
$$

然后直接最小化：

$$
\mathbb E\|F_\theta(X,\sigma)-Y\|^2
$$

理论上，只要网络容量足够、训练充分，这样完全可以学习最优 Denoiser。

但问题在于不同噪声水平下，网络面对的输入尺度和目标函数差异很大。

假设真实数据已经中心化：

$$
\mathbb E[Y]=0
$$

并且每个维度具有典型方差：

$$
\operatorname{Var}(Y_j)=\sigma_{\mathrm{data}}^2
\tag{6.7}
$$

这里的 $\sigma_{\mathrm{data}}$ 表示真实数据的典型标准差，不是加噪标准差 $\sigma$。

由于：

$$
X=Y+\sigma Z
$$

且 $Y$ 与 $Z$ 独立：

$$
\boxed{
\operatorname{Var}(X_j)
=
\sigma_{\mathrm{data}}^2+\sigma^2
}
\tag{6.8}
$$

于是：

### 低噪声情况

当：

$$
\sigma\ll\sigma_{\mathrm{data}}
$$

有：

$$
X\approx Y
$$

因此：

$$
D^*(X;\sigma)\approx X
$$

网络主要需要学习接近恒等映射的函数。

### 高噪声情况

当：

$$
\sigma\gg\sigma_{\mathrm{data}}
$$

有：

$$
X\approx\sigma Z
$$

输入幅度很大，但其中关于原始数据的信息很弱。

此时：

$$
D^*(x;\sigma)
$$

主要依赖数据分布的统计结构。

所以，直接训练：

$$
F_\theta(X,\sigma)\rightarrow Y
$$

会让同一个网络面对不同尺度的输入，以及结构差异很大的映射关系。

EDM 因此提出：

$$
\boxed{
D_\theta(x;\sigma)
=
c_{\mathrm{skip}}(\sigma)x
+
c_{\mathrm{out}}(\sigma)
F_\theta
\left(
c_{\mathrm{in}}(\sigma)x,
c_{\mathrm{noise}}(\sigma)
\right)
}
\tag{6.9}
$$

这就是网络预条件化。

其目的不是改变最终预测对象，而是**重新组织内部网络需要学习的函数**。

---

## 6.3 从残差学习理解网络预条件化

先忽略输入缩放和输出缩放，只考虑：

$$
\boxed{
D_\theta(X;\sigma)
=
cX+R_\theta(X,\sigma)
}
\tag{6.10}
$$

其中：

$$
c=c(\sigma)
$$

是一个只依赖噪声水平的标量。

对于任意真实样本，都有恒等式：

$$
\boxed{
Y=cX+(Y-cX)
}
\tag{6.11}
$$

因此，如果希望：

$$
D_\theta(X;\sigma)\approx Y
$$

自然可以让：

$$
R_\theta(X,\sigma)
$$

去拟合残差标签：

$$
\boxed{
R_{\mathrm{target}}=Y-cX
}
\tag{6.12}
$$

需要注意：

$$
R_{\mathrm{target}}
$$

仍然依赖真实的 $Y$。

而网络只能观察到 $X$。

因此平方损失下的最优残差函数是：

$$
R^*(x,\sigma)
=
\mathbb E[Y-cX\mid X=x]
$$

因为 $c$ 只依赖 $\sigma$，所以：

$$
R^*(x,\sigma)
=
\mathbb E[Y\mid X=x]-cx
$$

于是：

$$
\begin{aligned}
D^*(x,\sigma)
&=
cx+R^*(x,\sigma)\\
&=
cx+\mathbb E[Y\mid X=x]-cx\\
&=
\mathbb E[Y\mid X=x]
\end{aligned}
$$

因此：

$$
\boxed{
D^*(x,\sigma)
=
cx+R^*(x,\sigma)
}
\tag{6.13}
$$

这说明：

**将目标拆成线性分支和残差分支，不会改变理论最优 Denoiser。**

它只是改变了网络内部学习任务的表示方式。

现在的问题变成：

> 应该如何选择 $c$，使网络需要学习的残差尽可能容易处理？

---

## 6.4 推导 $c_{\mathrm{skip}}$：最优线性预测

EDM 希望线性分支：

$$
c_{\mathrm{skip}}X
$$

能够尽可能接近真实数据 $Y$。

因此选择：

$$
\boxed{
c_{\mathrm{skip}}
=
\arg\min_c
\mathbb E\|Y-cX\|^2
}
\tag{6.14}
$$

这个优化问题只在线性函数：

$$
cX
$$

的范围内寻找最优预测。

它并不等价于直接求：

$$
\mathbb E[Y\mid X]
$$

因为后者通常是非线性函数。

### 第一步：展开残差

由：

$$
X=Y+\sigma Z
$$

得到：

$$
\begin{aligned}
Y-cX
&=
Y-c(Y+\sigma Z)\\
&=
(1-c)Y-c\sigma Z
\end{aligned}
\tag{6.15}
$$

### 第二步：计算残差均方误差

先考虑单个维度：

$$
J(c)
=
\mathbb E[(Y-cX)^2]
$$

代入：

$$
J(c)
=
\mathbb E[
((1-c)Y-c\sigma Z)^2
]
$$

展开：

$$
\begin{aligned}
J(c)
&=
(1-c)^2\mathbb E[Y^2]\\
&\quad+
c^2\sigma^2\mathbb E[Z^2]\\
&\quad-
2c(1-c)\sigma\mathbb E[YZ]
\end{aligned}
\tag{6.16}
$$

根据假设：

$$
\mathbb E[Y]=0
$$

$$
\mathbb E[Y^2]=\sigma_{\mathrm{data}}^2
$$

$$
\mathbb E[Z^2]=1
$$

$$
Y\perp Z
$$

因此：

$$
\mathbb E[YZ]=0
$$

得到：

$$
\boxed{
J(c)
=
(1-c)^2\sigma_{\mathrm{data}}^2
+
c^2\sigma^2
}
\tag{6.17}
$$

### 第三步：求最优系数

对 $c$ 求导：

$$
\frac{dJ}{dc}
=
-2(1-c)\sigma_{\mathrm{data}}^2
+
2c\sigma^2
$$

令：

$$
\frac{dJ}{dc}=0
$$

得到：

$$
(1-c)\sigma_{\mathrm{data}}^2
=
c\sigma^2
$$

整理：

$$
\sigma_{\mathrm{data}}^2
=
c(\sigma_{\mathrm{data}}^2+\sigma^2)
$$

所以：

$$
\boxed{
c_{\mathrm{skip}}(\sigma)
=
\frac{\sigma_{\mathrm{data}}^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
}
\tag{6.18}
$$

### 第四步：理解这个系数

当：

$$
\sigma\rightarrow0
$$

有：

$$
c_{\mathrm{skip}}\rightarrow1
$$

所以：

$$
D_\theta(X;\sigma)
\approx X+\text{残差}
$$

这是合理的，因为低噪声状态本来就接近真实数据。

当：

$$
\sigma\rightarrow\infty
$$

有：

$$
c_{\mathrm{skip}}\rightarrow0
$$

所以：

$$
D_\theta(X;\sigma)
\approx\text{网络预测}
$$

这是合理的，因为高噪声状态不能直接当作真实数据使用。

因此：

$$
\boxed{
c_{\mathrm{skip}}
\text{ 根据噪声水平，自适应调整输入直通分支的权重}
}
$$

### 一个重要的数学区别

$$
c_{\mathrm{skip}}X
$$

是给定假设下的最优标量线性估计。

但：

$$
D^*(X;\sigma)
=
\mathbb E[Y\mid X]
$$

是最优非线性均方误差估计。

通常：

$$
\boxed{
c_{\mathrm{skip}}X
\neq
D^*(X;\sigma)
}
\tag{6.19}
$$

只有在某些特殊分布下，例如零均值各向同性高斯数据，二者才会一致。

对于真实图像、轨迹等非高斯数据，网络仍然需要学习非线性残差。

---

## 6.5 推导 $c_{\mathrm{out}}$：让残差监督目标具有稳定尺度

现在有：

$$
D_\theta(X;\sigma)
=
c_{\mathrm{skip}}X+R_\theta(X,\sigma)
$$

残差监督目标：

$$
R_{\mathrm{target}}
=
Y-c_{\mathrm{skip}}X
\tag{6.20}
$$

虽然已经选择了最优线性系数，但残差的尺度仍然依赖于噪声水平。

EDM 因此进一步定义：

$$
\boxed{
R_\theta
=
c_{\mathrm{out}}F_\theta
}
\tag{6.21}
$$

于是：

$$
\boxed{
D_\theta
=
c_{\mathrm{skip}}X
+
c_{\mathrm{out}}F_\theta
}
\tag{6.22}
$$

对应的内部网络监督标签为：

$$
\boxed{
F_{\mathrm{target}}
=
\frac{
Y-c_{\mathrm{skip}}X
}{
c_{\mathrm{out}}
}
}
\tag{6.23}
$$

EDM 希望：

$$
\boxed{
\operatorname{Var}(F_{\mathrm{target}})=1
}
\tag{6.24}
$$

这里仍然采用每个数据维度具有相同典型方差的近似假设。

### 第一步：计算原始残差的方差

根据：

$$
Y-c_{\mathrm{skip}}X
=
(1-c_{\mathrm{skip}})Y
-
c_{\mathrm{skip}}\sigma Z
$$

且 $Y,Z$ 独立，得到：

$$
\begin{aligned}
\operatorname{Var}(R_{\mathrm{target}})
&=
(1-c_{\mathrm{skip}})^2
\sigma_{\mathrm{data}}^2\\
&\quad+
c_{\mathrm{skip}}^2\sigma^2
\end{aligned}
\tag{6.25}
$$

### 第二步：代入 $c_{\mathrm{skip}}$

已知：

$$
c_{\mathrm{skip}}
=
\frac{\sigma_{\mathrm{data}}^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
$$

因此：

$$
1-c_{\mathrm{skip}}
=
\frac{\sigma^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
$$

代入：

$$
\begin{aligned}
\operatorname{Var}(R_{\mathrm{target}})
&=
\frac{
\sigma^4\sigma_{\mathrm{data}}^2
}{
(\sigma_{\mathrm{data}}^2+\sigma^2)^2
}\\
&\quad+
\frac{
\sigma_{\mathrm{data}}^4\sigma^2
}{
(\sigma_{\mathrm{data}}^2+\sigma^2)^2
}
\end{aligned}
$$

提取公因子：

$$
=
\frac{
\sigma^2\sigma_{\mathrm{data}}^2
(\sigma^2+\sigma_{\mathrm{data}}^2)
}{
(\sigma_{\mathrm{data}}^2+\sigma^2)^2
}
$$

约分：

$$
\boxed{
\operatorname{Var}(R_{\mathrm{target}})
=
\frac{
\sigma^2\sigma_{\mathrm{data}}^2
}{
\sigma_{\mathrm{data}}^2+\sigma^2
}
}
\tag{6.26}
$$

### 第三步：确定输出缩放系数

因为：

$$
F_{\mathrm{target}}
=
\frac{R_{\mathrm{target}}}{c_{\mathrm{out}}}
$$

希望：

$$
\operatorname{Var}(F_{\mathrm{target}})=1
$$

所以：

$$
c_{\mathrm{out}}^2
=
\operatorname{Var}(R_{\mathrm{target}})
$$

得到：

$$
\boxed{
c_{\mathrm{out}}(\sigma)
=
\frac{
\sigma\sigma_{\mathrm{data}}
}{
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
}
\tag{6.27}
$$

### 第四步：理解它的作用

当：

$$
\sigma\ll\sigma_{\mathrm{data}}
$$

有：

$$
c_{\mathrm{out}}\approx\sigma
$$

此时网络的输出修正被缩小：

$$
c_{\mathrm{out}}F_\theta\approx\sigma F_\theta
$$

因此低噪声状态下：

$$
D_\theta\approx X+\sigma F_\theta
$$

当：

$$
\sigma\gg\sigma_{\mathrm{data}}
$$

有：

$$
c_{\mathrm{out}}\approx\sigma_{\mathrm{data}}
$$

因此网络输出被转换回真实数据的典型尺度。

核心作用是：

$$
\boxed{
\text{让内部网络学习的残差目标在不同噪声水平下具有统一的典型方差}
}
$$

---

## 6.6 推导 $c_{\mathrm{in}}$：输入归一化

前面已经得到：

$$
\operatorname{Var}(X)
=
\sigma_{\mathrm{data}}^2+\sigma^2
$$

不同噪声水平下，网络输入尺度不同。

EDM 因此定义：

$$
\boxed{
\tilde X=c_{\mathrm{in}}X
}
\tag{6.28}
$$

希望：

$$
\operatorname{Var}(\tilde X)=1
$$

所以：

$$
c_{\mathrm{in}}^2
(\sigma_{\mathrm{data}}^2+\sigma^2)=1
$$

得到：

$$
\boxed{
c_{\mathrm{in}}(\sigma)
=
\frac1{
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
}
\tag{6.29}
$$

它的作用很直接：

$$
\boxed{
\text{归一化不同噪声水平下的网络输入尺度}
}
$$

与：

$$
c_{\mathrm{skip}}
$$

和：

$$
c_{\mathrm{out}}
$$

不同，$c_{\mathrm{in}}$ 不直接决定残差分解，而是改善网络输入的数值条件。

---

## 6.7 $c_{\mathrm{noise}}$：噪声水平编码

网络除了需要知道带噪输入：

$$
X
$$

还需要知道当前噪声水平：

$$
\sigma
$$

因为相同的 $X$ 在不同 $\sigma$ 下对应不同的后验分布：

$$
p(Y\mid X,\sigma)
$$

因此 Denoiser 必须具有噪声条件：

$$
D_\theta(X;\sigma)
$$

EDM 使用：

$$
\boxed{
c_{\mathrm{noise}}(\sigma)
=
\frac14\ln\sigma
}
\tag{6.30}
$$

将其作为内部网络的噪声条件输入。

使用：

$$
\ln\sigma
$$

可以压缩跨越多个数量级的噪声范围。

而：

$$
\frac14
$$

是经验性尺度设计，并不是由前面的最优线性估计推导得到的。

---

## 6.8 最终网络预条件化表达式

将四个系数组合：

$$
\boxed{
D_\theta(x;\sigma)
=
c_{\mathrm{skip}}(\sigma)x
+
c_{\mathrm{out}}(\sigma)
F_\theta
\left(
c_{\mathrm{in}}(\sigma)x,
c_{\mathrm{noise}}(\sigma)
\right)
}
\tag{6.31}
$$

其中：

$$
\boxed{
c_{\mathrm{skip}}
=
\frac{\sigma_{\mathrm{data}}^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
}
$$

$$
\boxed{
c_{\mathrm{out}}
=
\frac{
\sigma\sigma_{\mathrm{data}}
}{
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
}
$$

$$
\boxed{
c_{\mathrm{in}}
=
\frac1{
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
}
$$

$$
\boxed{
c_{\mathrm{noise}}
=
\frac14\ln\sigma
}
$$

四个系数的作用：

| 系数 | 设计目标 | 直接作用 |
|---|---|---|
| $c_{\mathrm{skip}}$ | 最小化线性预测残差的均方误差 | 构造最优标量线性基线 |
| $c_{\mathrm{out}}$ | 归一化残差监督目标 | 稳定内部网络输出目标尺度 |
| $c_{\mathrm{in}}$ | 归一化带噪输入 | 稳定内部网络输入尺度 |
| $c_{\mathrm{noise}}$ | 表达当前噪声水平 | 提供噪声条件 |

最重要的理解是：

$$
\boxed{
\text{预条件化不是改变 Denoiser 的最优目标，而是改变网络学习该目标的方式}
}
$$

---

## 6.9 内部网络 $F_\theta$ 实际预测什么？

这部分需要进一步展开，因为它关系到 EDM 与噪声预测、数据预测和 $v$-prediction 的联系。

前面已经得到：

$$
\boxed{
F_{\mathrm{target}}
=
\frac{
Y-c_{\mathrm{skip}}X
}{
c_{\mathrm{out}}
}
}
\tag{6.32}
$$

代入：

$$
X=Y+\sigma Z
$$

得到：

$$
F_{\mathrm{target}}
=
\frac{
(1-c_{\mathrm{skip}})Y
-
c_{\mathrm{skip}}\sigma Z
}{
c_{\mathrm{out}}
}
\tag{6.33}
$$

现在代入三个系数。

首先：

$$
1-c_{\mathrm{skip}}
=
\frac{\sigma^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
$$

所以分子：

$$
\begin{aligned}
Y-c_{\mathrm{skip}}X
&=
\frac{\sigma^2}{\sigma_{\mathrm{data}}^2+\sigma^2}Y\\
&\quad-
\frac{\sigma_{\mathrm{data}}^2\sigma}
{\sigma_{\mathrm{data}}^2+\sigma^2}Z
\end{aligned}
$$

提取：

$$
\frac{\sigma}{\sigma_{\mathrm{data}}^2+\sigma^2}
$$

得到：

$$
Y-c_{\mathrm{skip}}X
=
\frac{\sigma}
{\sigma_{\mathrm{data}}^2+\sigma^2}
\left(
\sigma Y-\sigma_{\mathrm{data}}^2Z
\right)
$$

又因为：

$$
c_{\mathrm{out}}
=
\frac{
\sigma\sigma_{\mathrm{data}}
}{
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
$$

因此：

$$
\boxed{
F_{\mathrm{target}}
=
\frac{
\sigma Y-\sigma_{\mathrm{data}}^2Z
}{
\sigma_{\mathrm{data}}
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
}
\tag{6.34}
$$

这就是内部网络实际学习的监督标签。

可以看到，它既不是纯粹的：

$$
Y
$$

也不是纯粹的：

$$
Z
$$

而是二者的线性组合。

### 低噪声极限

当：

$$
\sigma\rightarrow0^+
$$

分母：

$$
\sigma_{\mathrm{data}}
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
\rightarrow
\sigma_{\mathrm{data}}^2
$$

分子：

$$
\sigma Y-\sigma_{\mathrm{data}}^2Z
\rightarrow
-\sigma_{\mathrm{data}}^2Z
$$

因此：

$$
\boxed{
F_{\mathrm{target}}\rightarrow-Z
}
\tag{6.35}
$$

低噪声时，内部网络的监督标签趋向负噪声。

### 高噪声极限

当：

$$
\sigma\rightarrow\infty
$$

分母近似为：

$$
\sigma_{\mathrm{data}}\sigma
$$

分子主要为：

$$
\sigma Y
$$

因此：

$$
\boxed{
F_{\mathrm{target}}
\rightarrow
\frac{Y}{\sigma_{\mathrm{data}}}
}
\tag{6.36}
$$

高噪声时，内部网络的监督标签趋向归一化后的真实数据。

因此：

$$
\boxed{
\text{EDM 内部网络的监督目标会随噪声水平，从负噪声目标逐渐过渡到数据目标}
}
$$

这里讨论的是逐样本监督标签的极限，不表示在极高噪声下网络能够准确恢复每个样本的真实 $Y$。

实际平方损失下，内部网络学到的仍然是该标签在给定输入条件下的条件期望。

---

## 6.10 原始 MSE 到底是在什么空间计算？

EDM 最初考虑的是：

$$
\boxed{
\mathcal L_D
=
\mathbb E
\left[
\|D_\theta(X;\sigma)-Y\|^2
\right]
}
\tag{6.37}
$$

这里的误差是在：

$$
\boxed{
x_0\text{ 数据空间}
}
$$

计算的。

不是 Score 空间，也不是噪声空间。

不过，这三种空间之间存在精确的代数关系。

### 数据预测与噪声预测

定义噪声预测器：

$$
\epsilon_\theta(X;\sigma)
$$

并令：

$$
\boxed{
D_\theta(X;\sigma)
=
X-\sigma\epsilon_\theta(X;\sigma)
}
\tag{6.38}
$$

由于：

$$
X=Y+\sigma Z
$$

因此：

$$
\begin{aligned}
D_\theta-Y
&=
X-\sigma\epsilon_\theta-Y\\
&=
\sigma Z-\sigma\epsilon_\theta
\end{aligned}
$$

得到：

$$
\boxed{
D_\theta-Y
=
\sigma(Z-\epsilon_\theta)
}
\tag{6.39}
$$

所以：

$$
\boxed{
\|D_\theta-Y\|^2
=
\sigma^2
\|\epsilon_\theta-Z\|^2
}
\tag{6.40}
$$

因此：

$$
\boxed{
\text{数据空间 MSE}
=
\sigma^2
\times
\text{噪声空间 MSE}
}
$$

### 数据预测与 Score 预测

根据 Tweedie's Formula：

$$
D^*(X;\sigma)
=
X+\sigma^2s^*(X;\sigma)
$$

定义模型 Score：

$$
s_\theta(X;\sigma)
$$

并令：

$$
\boxed{
D_\theta(X;\sigma)
=
X+\sigma^2s_\theta(X;\sigma)
}
\tag{6.41}
$$

因此：

$$
\begin{aligned}
D_\theta-Y
&=
X+\sigma^2s_\theta-Y\\
&=
\sigma Z+\sigma^2s_\theta
\end{aligned}
$$

提取：

$$
\sigma^2
$$

得到：

$$
D_\theta-Y
=
\sigma^2
\left(
s_\theta+\frac{Z}{\sigma}
\right)
$$

所以：

$$
\boxed{
\|D_\theta-Y\|^2
=
\sigma^4
\left\|
s_\theta+\frac{Z}{\sigma}
\right\|^2
}
\tag{6.42}
$$

因此：

$$
\boxed{
\text{数据空间 MSE}
=
\sigma^4
\times
\text{Denoising Score Matching MSE}
}
$$

这些等式说明：

**虽然数据预测、噪声预测和 Score 预测之间可以相互转换，但它们在使用不加权 MSE 时，对不同噪声水平施加的优化权重并不相同。**

---

## 6.11 EDM 为什么引入 Loss 权重 $\lambda(\sigma)$？

EDM 的训练目标为：

$$
\boxed{
\mathcal L_{\mathrm{EDM}}
=
\mathbb E_{\sigma,Y,Z}
\left[
\lambda(\sigma)
\|D_\theta(X;\sigma)-Y\|^2
\right]
}
\tag{6.43}
$$

其中：

$$
X=Y+\sigma Z
$$

现在代入预条件化表达式：

$$
D_\theta
=
c_{\mathrm{skip}}X+c_{\mathrm{out}}F_\theta
$$

于是：

$$
D_\theta-Y
=
c_{\mathrm{skip}}X+c_{\mathrm{out}}F_\theta-Y
$$

重新整理：

$$
D_\theta-Y
=
c_{\mathrm{out}}F_\theta
-
(Y-c_{\mathrm{skip}}X)
$$

定义：

$$
F_{\mathrm{target}}
=
\frac{
Y-c_{\mathrm{skip}}X
}{
c_{\mathrm{out}}
}
$$

得到：

$$
\boxed{
D_\theta-Y
=
c_{\mathrm{out}}
(F_\theta-F_{\mathrm{target}})
}
\tag{6.44}
$$

平方：

$$
\boxed{
\|D_\theta-Y\|^2
=
c_{\mathrm{out}}^2
\|F_\theta-F_{\mathrm{target}}\|^2
}
\tag{6.45}
$$

代入训练目标：

$$
\boxed{
\mathcal L_{\mathrm{EDM}}
=
\mathbb E
\left[
\lambda(\sigma)c_{\mathrm{out}}^2
\|F_\theta-F_{\mathrm{target}}\|^2
\right]
}
\tag{6.46}
$$

现在问题变成：

> 既然已经将内部网络监督标签归一化，是否还需要让不同噪声水平的误差乘上不同的额外尺度系数？

EDM 选择：

$$
\boxed{
\lambda(\sigma)c_{\mathrm{out}}^2=1
}
\tag{6.47}
$$

因此：

$$
\lambda(\sigma)
=
\frac1{c_{\mathrm{out}}^2}
$$

由于：

$$
c_{\mathrm{out}}^2
=
\frac{
\sigma^2\sigma_{\mathrm{data}}^2
}{
\sigma_{\mathrm{data}}^2+\sigma^2
}
$$

得到：

$$
\boxed{
\lambda(\sigma)
=
\frac{
\sigma_{\mathrm{data}}^2+\sigma^2
}{
\sigma^2\sigma_{\mathrm{data}}^2
}
}
\tag{6.48}
$$

最终：

$$
\boxed{
\mathcal L_{\mathrm{EDM}}
=
\mathbb E
\left[
\|F_\theta-F_{\mathrm{target}}\|^2
\right]
}
\tag{6.49}
$$

这里最重要的结论是：

$$
\boxed{
\lambda(\sigma)\neq1
}
$$

而是：

$$
\boxed{
\lambda(\sigma)c_{\mathrm{out}}^2=1
}
$$

也就是说：

**在最终 Denoiser 的数据空间中，Loss 仍然是加权 MSE；但转换到内部网络的归一化监督空间后，Loss 变成普通 MSE。**

这正是 EDM 网络预条件化与 Loss 权重之间的联系。

---

## 6.12 为什么这种 Loss 设计更容易训练？

假设存在两个噪声水平：

$$
\sigma_a\ll\sigma_b
$$

如果直接训练：

$$
D_\theta(X;\sigma)\rightarrow Y
$$

那么不同噪声水平下，网络输入的尺度可能相差很大。

与此同时，网络需要学习的目标函数结构也不同。

EDM 依次进行三个操作。

### 第一：输入尺度归一化

$$
\boxed{
\operatorname{Var}(c_{\mathrm{in}}X)=1
}
\tag{6.50}
$$

使不同噪声水平下，网络输入具有统一的典型尺度。

### 第二：输出目标归一化

$$
\boxed{
\operatorname{Var}(F_{\mathrm{target}})=1
}
\tag{6.51}
$$

使不同噪声水平下，内部网络监督标签具有统一的典型尺度。

### 第三：消除额外的 Loss 尺度系数

$$
\boxed{
\lambda c_{\mathrm{out}}^2=1
}
\tag{6.52}
$$

使内部网络直接最小化：

$$
\|F_\theta-F_{\mathrm{target}}\|^2
$$

因此：

$$
\boxed{
\text{输入尺度稳定}
+
\text{监督目标尺度稳定}
+
\text{有效 Loss 系数统一}
}
$$

这有利于优化器处理不同噪声水平下的训练样本。

但需要注意：

**这并不意味着不同噪声水平下的梯度范数严格相等。**

因为：

$$
\nabla_\theta\mathcal L
=
2(F_\theta-F_{\mathrm{target}})
\nabla_\theta F_\theta
\tag{6.53}
$$

仍然依赖：

- 预测误差。
- 网络 Jacobian。
- 数据分布。
- 参数状态。
- 不同噪声水平的任务难度。

所以 EDM 实现的是输入和监督尺度上的规范化，而不是严格的梯度均衡。

---

## 6.13 为什么还需要设计训练噪声分布？

即使：

$$
\lambda c_{\mathrm{out}}^2=1
$$

也不意味着所有噪声水平都应该以相同概率参与训练。

因为不同噪声水平对应不同的去噪任务。

### 极低噪声

当：

$$
\sigma\rightarrow0
$$

有：

$$
X\approx Y
$$

Denoiser 接近恒等映射。

### 极高噪声

当：

$$
\sigma\rightarrow\infty
$$

有：

$$
X\approx\sigma Z
$$

输入包含的样本特定信息很少。

### 中间噪声

在中间噪声区域，输入仍然保留部分数据结构，但又需要网络恢复缺失的信息。

因此，训练资源如何分配到不同噪声区域，会影响模型最终表现。

EDM 使用：

$$
\boxed{
\ln\sigma
\sim
\mathcal N(P_{\mathrm{mean}},P_{\mathrm{std}}^2)
}
\tag{6.54}
$$

也就是：

$$
\boxed{
\sigma
=
\exp(P_{\mathrm{mean}}+P_{\mathrm{std}}Z)
}
\tag{6.55}
$$

其中：

$$
Z\sim\mathcal N(0,1)
$$

注意，这里的 $Z$ 是采样噪声水平所用的标准高斯随机变量；实现中可以与加噪用的高斯变量分别采样。

论文的代表性推荐设置为：

$$
\boxed{
P_{\mathrm{mean}}=-1.2,\qquad
P_{\mathrm{std}}=1.2
}
\tag{6.56}
$$

对应的概率密度：

$$
\boxed{
p_{\mathrm{train}}(\sigma)
=
\frac1{
\sigma P_{\mathrm{std}}\sqrt{2\pi}
}
\exp
\left[
-\frac{
(\ln\sigma-P_{\mathrm{mean}})^2
}{
2P_{\mathrm{std}}^2
}
\right]
}
\tag{6.57}
$$

### 为什么使用对数正态分布？

因为噪声标准差通常跨越多个数量级。

例如：

$$
\sigma_{\min}=0.002
$$

$$
\sigma_{\max}=80
$$

如果直接在：

$$
[0.002,80]
$$

上均匀采样，大量训练样本会落在较高的绝对噪声数值区间。

但这些区域不一定是最值得分配训练资源的地方。

通过在：

$$
\ln\sigma
$$

空间采样，可以覆盖多个数量级，同时让采样概率集中在特定噪声区域。

这里的：

$$
P_{\mathrm{mean}},P_{\mathrm{std}}
$$

控制训练资源分配。

它们不是由 PF ODE 数学推导唯一确定的，而是经验性超参数。

---

## 6.14 Loss 权重与噪声采样分布的区别

完整训练目标可以写为：

$$
\boxed{
\mathcal L
=
\int_0^\infty
p_{\mathrm{train}}(\sigma)
\lambda(\sigma)
L_D(\sigma)\,d\sigma
}
\tag{6.58}
$$

其中：

$$
L_D(\sigma)
=
\mathbb E_{Y,Z}
\left[
\|D_\theta(Y+\sigma Z;\sigma)-Y\|^2
\right]
$$

这里：

$$
p_{\mathrm{train}}(\sigma)
$$

决定噪声水平被采样的频率。

而：

$$
\lambda(\sigma)
$$

决定对应样本的 Loss 权重。

从期望目标角度看，二者都影响不同噪声区域的贡献。

但在随机优化过程中，两者并不完全等价。

例如：

**方案 A：提高某个噪声区间的采样概率。**

优化器会更频繁地看到该区间的训练样本。

**方案 B：提高某个噪声区间的 Loss 权重。**

优化器看到该区间样本的频率不变，但单次样本产生的梯度被放大。

因此，两种方案可能具有不同的梯度方差和训练稳定性。

EDM 的思想是：

$$
\boxed{
\text{噪声采样分布}
+
\text{Loss 权重}
}
$$

需要联合设计，而不是只调整其中一个。

---

## 6.15 Non-Leaky Augmentation

EDM 还研究了数据增强。

假设真实数据为：

$$
Y
$$

对其施加增强变换：

$$
\boxed{
\tilde Y=A_\phi(Y)
}
\tag{6.59}
$$

其中：

$$
\phi
$$

表示增强参数。

然后构造带噪输入：

$$
X=\tilde Y+\sigma Z
$$

如果不提供增强信息，网络可能直接学习增强后的数据分布。

因此 EDM 将增强参数也作为条件输入：

$$
\boxed{
D_\theta(X;\sigma,\phi)
}
\tag{6.60}
$$

推理时使用表示无增强的条件：

$$
\phi=0
$$

这样可以利用数据增强改善泛化能力，同时尽量避免生成结果受到训练增强分布的影响。

这部分主要是针对图像生成任务的训练技巧。

它不是 EDM 的 PF ODE 或预条件化数学框架所必需的组成部分。

---

## 6.16 Section 5 总结

EDM 的训练部分可以归纳为：

$$
\boxed{
\text{原始数据预测}
\rightarrow
\text{最优线性基线}
\rightarrow
\text{归一化残差学习}
}
$$

最重要的公式是：

$$
\boxed{
D_\theta
=
c_{\mathrm{skip}}X
+
c_{\mathrm{out}}
F_\theta(c_{\mathrm{in}}X,c_{\mathrm{noise}})
}
$$

其中：

$$
\boxed{
c_{\mathrm{skip}}
=
\frac{\sigma_{\mathrm{data}}^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
}
$$

$$
\boxed{
c_{\mathrm{out}}
=
\frac{\sigma\sigma_{\mathrm{data}}}
{\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}}
}
$$

$$
\boxed{
c_{\mathrm{in}}
=
\frac1{\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}}
}
$$

$$
\boxed{
c_{\mathrm{noise}}
=
\frac14\ln\sigma
}
$$

以及：

$$
\boxed{
\lambda(\sigma)
=
\frac1{c_{\mathrm{out}}^2}
}
$$

最终使：

$$
\boxed{
\mathcal L_{\mathrm{EDM}}
=
\mathbb E
\left[
\|F_\theta-F_{\mathrm{target}}\|^2
\right]
}
$$

其中：

$$
\boxed{
F_{\mathrm{target}}
=
\frac{
Y-c_{\mathrm{skip}}X
}{
c_{\mathrm{out}}
}
}
$$

这一节最重要的结论是：

**EDM 并没有改变最优 Denoiser 的数学定义，而是通过合理的函数分解和尺度归一化，改善神经网络在不同噪声水平下的学习条件。**

---

# 七、Section 6：实验结果与论文结论

前面介绍的 ODE 参数化、Heun、非均匀时间网格、Stochastic Churn 和网络预条件化，都需要通过实验判断是否真正有效。

EDM 的一个重要特点是采用模块化消融实验，而不是只展示最终模型的生成质量。

## 7.1 实验设置与评价指标

论文主要使用以下数据集：

| 数据集 | 任务 |
|---|---|
| CIFAR-10 | $32\times32$ 图像生成 |
| ImageNet-64 | $64\times64$ 类别条件图像生成 |

主要评价指标包括：

**FID（Fréchet Inception Distance）**

衡量生成图像分布与真实图像分布在 Inception 特征空间中的差异。

通常：

$$
\boxed{\mathrm{FID}\downarrow}
$$

表示生成质量更好。

**NFE（Number of Function Evaluations）**

表示采样过程中调用神经网络的次数。

通常：

$$
\boxed{\mathrm{NFE}\downarrow}
$$

表示推理计算量更低。

因此论文关注的不只是最低 FID，还包括：

$$
\boxed{
\text{FID 与 NFE 之间的权衡}
}
$$

---

## 7.2 实验一：不重新训练模型，只改进采样器

EDM 首先使用已有的预训练扩散模型，保持网络权重不变，仅调整采样过程。

这些模型包括：

- VP 类模型。
- VE 类模型。
- iDDPM/ADM 类模型。

主要考察：

1. 将已有模型转换到统一采样框架。
2. 调整 ODE 时间参数化。
3. 使用二阶 Heun。
4. 采用非均匀噪声网格。
5. 比较不同 NFE 下的生成质量。

这一实验的重要性在于：

$$
\boxed{
\text{模型权重不变}
}
$$

因此性能变化主要来自采样过程。

论文发现，通过合理的采样器设计，已有预训练模型也能够获得明显改善。

例如，论文报告对已有 ImageNet-64 模型进行采样改进后，FID 从：

$$
2.07
$$

改善至：

$$
1.55
$$

这说明：

**扩散模型的生成质量不仅取决于网络训练，还受到数值采样方法的显著影响。**

但不能由此得出“采样器比模型训练更重要”的结论，因为最终效果仍然受到网络精度限制。

---

## 7.3 实验二：Euler、Heun 与采样网格

论文研究不同求解器与时间网格对采样质量的影响。

### Euler 的特点

每步只需要一次网络调用：

$$
\mathrm{NFE}=1
$$

但局部误差：

$$
O(h^2)
$$

### Heun 的特点

每步通常需要两次网络调用：

$$
\mathrm{NFE}=2
$$

但局部误差：

$$
O(h^3)
$$

因此，在相同积分步数下，Heun 往往具有更高的数值精度。

但论文真正关心的是：

$$
\boxed{
\text{相同 NFE 下，哪种方法的生成质量更好？}
}
$$

实验结果支持二阶 Heun 是一种较好的效率与精度折中。

同时，非均匀网格：

$$
t_i
=
\left[
t_{\max}^{1/\rho}
+
\frac{i}{N-1}
(t_{\min}^{1/\rho}-t_{\max}^{1/\rho})
\right]^\rho
$$

能够更合理地分配积分步长。

论文推荐：

$$
\boxed{\rho=7}
$$

但这是经验性选择，而不是从 ODE 局部误差公式中严格求出的全局最优参数。

---

## 7.4 实验三：随机性是否能够提高生成质量？

论文比较了不同：

$$
S_{\mathrm{churn}}
$$

下的采样效果。

其中：

$$
S_{\mathrm{churn}}=0
$$

对应确定性采样。

当：

$$
S_{\mathrm{churn}}>0
$$

时，采样过程中会注入随机扰动。

论文发现：

- 对部分原始训练模型，适量随机性能够明显改善 FID。
- 对采用 EDM 改进训练方案的部分 CIFAR-10 模型，确定性采样反而更好。
- 对 ImageNet-64 等更复杂的任务，适量随机性仍然可能带来收益。
- 过强的随机扰动可能导致细节损失或颜色偏移。

因此：

$$
\boxed{
\text{随机性是否有利，取决于模型与任务}
}
$$

一个值得进一步研究的解释是：

当网络预测误差较大时，随机采样可能缓解某些累积误差。

当网络已经足够准确时，额外随机扰动可能反而破坏生成质量。

但这只是与实验现象一致的机制解释，不能将其理解为经过严格证明的普遍规律。

---

## 7.5 实验四：网络预条件化与训练策略

EDM 通过逐步加入不同训练改进，研究它们对生成质量的影响。

主要包括：

- 网络结构和容量调整。
- 网络预条件化。
- Loss 权重设计。
- 训练噪声分布。
- 数据增强。

这些实验的重要结论是：

**训练改进需要作为整体进行理解。**

例如，网络预条件化能够让：

$$
F_{\mathrm{target}}
$$

具有相对稳定的尺度。

但如果训练噪声分布不合理，网络仍然可能将过多训练资源分配到某些不重要的噪声区间。

同样，即使选择了合理的噪声采样分布，如果不同噪声水平下的 Loss 尺度差异很大，训练过程仍然可能不稳定。

因此，EDM 的训练方案可以理解为：

$$
\boxed{
\text{Preconditioning}
+
\text{Loss Weighting}
+
\text{Noise Distribution}
}
\tag{7.1}
$$

三者共同发挥作用。

---

## 7.6 论文最终结果

EDM 在论文发表时报告了以下代表性结果：

| 数据集 | 任务 | FID |
|---|---|---:|
| CIFAR-10 | 类别条件生成 | 1.79 |
| CIFAR-10 | 无条件生成 | 1.97 |
| ImageNet-64 | 类别条件生成 | 1.36 |

这些结果说明，EDM 通过系统优化训练与采样设计，在当时取得了很强的生成效果。

但需要注意：

**这些是 NeurIPS 2022 论文中的实验结果，不代表当前生成模型的最新最佳成绩。**

---

# 八、EDM 与 Flow Matching 的数学联系

这一部分不是 EDM 正文的主要研究内容，而是基于前面的数学框架作出的进一步分析。

考虑高斯加噪路径：

$$
\boxed{
X_t=Y+tZ
}
\tag{8.1}
$$

其中：

$$
Y\sim p_{\mathrm{data}}
$$

$$
Z\sim\mathcal N(0,I)
$$

且二者独立。

对于固定的：

$$
(Y,Z)
$$

有：

$$
\boxed{
\frac{dX_t}{dt}=Z
}
\tag{8.2}
$$

因此条件路径是一条直线。

但 Flow Matching 最终需要学习的是只依赖当前状态和时间的边缘速度场：

$$
\boxed{
v^*(x,t)
=
\mathbb E[Z\mid X_t=x]
}
\tag{8.3}
$$

由于：

$$
Z=\frac{X_t-Y}{t}
$$

所以：

$$
\begin{aligned}
v^*(x,t)
&=
\mathbb E
\left[
\frac{X_t-Y}{t}
\mid X_t=x
\right]\\
&=
\frac{
x-\mathbb E[Y\mid X_t=x]
}{t}
\end{aligned}
$$

根据最优 Denoiser 的定义：

$$
D^*(x;t)
=
\mathbb E[Y\mid X_t=x]
$$

得到：

$$
\boxed{
v^*(x,t)
=
\frac{x-D^*(x;t)}{t}
}
\tag{8.4}
$$

而 EDM 的 PF ODE 为：

$$
\boxed{
\frac{dx}{dt}
=
\frac{x-D^*(x;t)}{t}
}
\tag{8.5}
$$

因此：

$$
\boxed{
v_{\mathrm{FM}}^*(x,t)
=
v_{\mathrm{EDM}}^*(x,t)
}
\tag{8.6}
$$

这说明：

**在相同的高斯加噪路径 $X_t=Y+tZ$ 下，Flow Matching 的最优边缘速度场与 EDM 的 PF ODE 速度场一致。**

但这不代表所有 Flow Matching 模型都等价于 EDM。

因为 Flow Matching 可以使用其他概率路径，例如：

$$
X_t=(1-t)Z+tY
$$

也可以使用不同的噪声耦合方式。

此外，还需要特别区分：

$$
\boxed{
\text{条件路径}
\neq
\text{边缘 ODE 轨迹}
}
$$

虽然：

$$
X_t=Y+tZ
$$

对于固定的 $Y,Z$ 是直线，但：

$$
v^*(x,t)
=
\mathbb E[Z\mid X_t=x]
$$

通常是一个非线性速度场。

因此边缘 ODE 的解轨迹一般不是直线。

这也解释了为什么 EDM 即使采用：

$$
\sigma(t)=t
$$

仍然需要多步数值积分。

---

# 九、EDM 与 Consistency Models 的区别

EDM 和 Consistency Models 都与 Probability Flow ODE 有关，但学习目标不同。

## 9.1 EDM 学习局部去噪信息

EDM 的最优 Denoiser：

$$
\boxed{
D^*(x,t)
=
\mathbb E[Y\mid X_t=x]
}
\tag{9.1}
$$

通过：

$$
\boxed{
v^*(x,t)
=
\frac{x-D^*(x,t)}{t}
}
\tag{9.2}
$$

构造 ODE 速度场。

因此：

$$
\boxed{
\text{EDM 学习 Denoiser，再通过 ODE 积分生成}
}
$$

## 9.2 Consistency Models 学习 ODE 终点映射

Consistency Models 希望学习：

$$
\boxed{
f^*(x_t,t)
=
\Phi_{t\rightarrow\epsilon}(x_t)
}
\tag{9.3}
$$

其中：

$$
\Phi_{t\rightarrow\epsilon}
$$

表示从当前状态沿 PF ODE 积分到边界时间 $\epsilon$ 的流映射。

因此：

$$
\boxed{
\text{Consistency Models 直接学习 ODE 终点映射}
}
$$

两者比较：

| 比较维度 | EDM | Consistency Models |
|---|---|---|
| 学习对象 | 条件去噪均值 | ODE 终点映射 |
| 数学关系 | 通过 Tweedie's Formula 得到速度场 | 通过一致性约束学习流映射 |
| 推理方式 | 通常多步积分 | 可以一步或少步生成 |
| 网络输出 | Denoiser | Consistency Function |
| 主要研究重点 | 训练与采样设计空间 | 少步生成与轨迹一致性 |

因此一般有：

$$
\boxed{
D^*(x,t)
\neq
\Phi_{t\rightarrow\epsilon}(x)
}
\tag{9.4}
$$

不能把 EDM 的 Denoiser 直接理解为一步生成器。

---

# 十、EDM 对自动驾驶轨迹规划的启发

结合 Conditional Flow Matching 轨迹规划，可以从 EDM 中提炼出三个比较有价值的研究方向。

## 10.1 采样器与训练目标应当分开研究

对于条件轨迹规划，假设：

$$
Y\in\mathbb R^{T\times d}
$$

表示未来轨迹。

场景条件：

$$
c
$$

来自感知、地图和导航信息。

Flow Matching 学习：

$$
\boxed{
v_\theta(x,t,c)
}
$$

然后通过 ODE：

$$
\frac{dx}{dt}
=
v_\theta(x,t,c)
$$

生成未来轨迹。

即使固定：

$$
v_\theta
$$

也可以独立研究：

- Euler。
- Heun。
- 不同时间网格。
- 不同 NFE。
- 不同初始噪声分布。

这与 EDM 的模块化研究思路一致。

尤其对于自动驾驶，不能只评价轨迹与真值的距离，还应同时关注：

$$
\boxed{
\text{碰撞率、舒适性、可行性、多模态覆盖率和推理延迟}
}
$$

因为更准确地求解 ODE，并不必然意味着规划轨迹更加安全。

---

## 10.2 网络预测空间与 Loss 空间可以分别设计

EDM 最值得借鉴的思想之一是：

$$
\boxed{
\text{最终输出目标}
\neq
\text{内部网络必须直接预测的目标}
}
$$

例如，最终需要：

$$
D_\theta(X,\sigma)\approx Y
$$

但内部网络实际学习：

$$
F_{\mathrm{target}}
=
\frac{
Y-c_{\mathrm{skip}}X
}{
c_{\mathrm{out}}
}
$$

这说明可以通过合理的函数分解，改善网络学习目标的尺度。

对于轨迹规划，也可以考虑：

$$
\boxed{
\text{最终轨迹}
=
\text{简单基线}
+
\text{网络预测残差}
}
$$

例如：

$$
\boxed{
Y_{\mathrm{pred}}
=
Y_{\mathrm{baseline}}
+
\Delta Y_\theta
}
\tag{10.1}
$$

其中基线可以来自：

- 匀速运动。
- 简单车辆运动学模型。
- 参考车道中心线。
- 其他低复杂度预测器。

网络只需要学习：

$$
\Delta Y_\theta
$$

但这里需要注意：

**EDM 的 $c_{\mathrm{skip}}$、$c_{\mathrm{out}}$ 是在特定高斯加噪假设下推导得到的，不能直接照搬到任意轨迹规划模型。**

对于 Conditional Flow Matching，需要根据实际概率路径、数据尺度和预测目标重新推导。

---

## 10.3 时间采样分布与 Loss 权重可以联合优化

Conditional Flow Matching 通常采用：

$$
t\sim U(0,1)
$$

并最小化：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E
\left[
\|v_\theta(X_t,t,c)-u_t\|^2
\right]
}
\tag{10.2}
$$

但不同时间区域的学习难度可能不同。

例如：

- 某些时间区域的速度目标方差较大。
- 某些时间区域对最终轨迹误差更敏感。
- 某些时间区域的速度场曲率更大。
- 某些时间区域可能更容易产生碰撞风险。

因此可以研究：

$$
\boxed{
t\sim p_{\mathrm{train}}(t)
}
$$

以及：

$$
\boxed{
\mathcal L
=
\mathbb E
\left[
w(t)
\|v_\theta(X_t,t,c)-u_t\|^2
\right]
}
\tag{10.3}
$$

这里：

$$
p_{\mathrm{train}}(t)
$$

控制时间采样频率。

而：

$$
w(t)
$$

控制不同时间区域的 Loss 权重。

这与 EDM 联合研究噪声分布和 Loss 权重的思想相似。

但最佳设计仍然需要根据具体任务进行理论分析和实验验证。

---

# 十一、全文核心公式汇总

## 11.1 高斯加噪分布

$$
\boxed{
X_\sigma=Y+\sigma Z
}
$$

其中：

$$
Y\sim p_{\mathrm{data}},\qquad
Z\sim\mathcal N(0,I)
$$

## 11.2 最优 Denoiser

$$
\boxed{
D^*(x;\sigma)
=
\mathbb E[Y\mid X_\sigma=x]
}
$$

## 11.3 Tweedie's Formula

$$
\boxed{
\nabla_x\log p_\sigma(x)
=
\frac{D^*(x;\sigma)-x}{\sigma^2}
}
$$

## 11.4 通用 Probability Flow ODE

在未缩放坐标 Score 下：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
s\sigma\dot\sigma
\nabla_u\log p_\sigma(u)
\bigg|_{u=x/s}
}
$$

在缩放后坐标 Score 下：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
s^2\sigma\dot\sigma
\nabla_x\log q_t(x)
}
$$

## 11.5 EDM 推荐 ODE

当：

$$
s(t)=1,\qquad\sigma(t)=t
$$

有：

$$
\boxed{
\frac{dx}{dt}
=
\frac{x-D_\theta(x;t)}{t}
}
$$

## 11.6 非均匀噪声网格

$$
\boxed{
t_i
=
\left[
t_{\max}^{1/\rho}
+
\frac{i}{N-1}
\left(
t_{\min}^{1/\rho}-t_{\max}^{1/\rho}
\right)
\right]^\rho
}
$$

推荐：

$$
\boxed{\rho=7}
$$

## 11.7 Heun

$$
\boxed{
\begin{aligned}
d_i&=v_\theta(x_i,t_i)\\
\tilde x_{i+1}&=x_i+h_id_i\\
\tilde d_{i+1}&=v_\theta(\tilde x_{i+1},t_{i+1})\\
x_{i+1}&=
x_i+\frac{h_i}{2}(d_i+\tilde d_{i+1})
\end{aligned}
}
$$

## 11.8 Stochastic Churn

$$
\boxed{
\hat t_i=(1+\gamma_i)t_i
}
$$

$$
\boxed{
\hat x_i
=
x_i+
\sqrt{\hat t_i^2-t_i^2}\,\epsilon_i
}
$$

其中：

$$
\epsilon_i\sim
\mathcal N(0,S_{\mathrm{noise}}^2I)
$$

然后从：

$$
\hat t_i
$$

使用 Heun 积分到：

$$
t_{i+1}
$$

## 11.9 网络预条件化

$$
\boxed{
D_\theta(x;\sigma)
=
c_{\mathrm{skip}}x
+
c_{\mathrm{out}}
F_\theta(c_{\mathrm{in}}x,c_{\mathrm{noise}})
}
$$

其中：

$$
\boxed{
c_{\mathrm{skip}}
=
\frac{\sigma_{\mathrm{data}}^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
}
$$

$$
\boxed{
c_{\mathrm{out}}
=
\frac{\sigma\sigma_{\mathrm{data}}}
{\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}}
}
$$

$$
\boxed{
c_{\mathrm{in}}
=
\frac1{\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}}
}
$$

$$
\boxed{
c_{\mathrm{noise}}
=
\frac14\ln\sigma
}
$$

## 11.10 内部网络监督目标

$$
\boxed{
F_{\mathrm{target}}
=
\frac{
Y-c_{\mathrm{skip}}X
}{
c_{\mathrm{out}}
}
}
$$

等价地：

$$
\boxed{
F_{\mathrm{target}}
=
\frac{
\sigma Y-\sigma_{\mathrm{data}}^2Z
}{
\sigma_{\mathrm{data}}
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
}
$$

## 11.11 Loss 权重

$$
\boxed{
\lambda(\sigma)
=
\frac{
\sigma^2+\sigma_{\mathrm{data}}^2
}{
\sigma^2\sigma_{\mathrm{data}}^2
}
}
$$

使：

$$
\boxed{
\lambda(\sigma)c_{\mathrm{out}}^2=1
}
$$

最终：

$$
\boxed{
\mathcal L_{\mathrm{EDM}}
=
\mathbb E
\left[
\|F_\theta-F_{\mathrm{target}}\|^2
\right]
}
$$

## 11.12 训练噪声分布

$$
\boxed{
\ln\sigma
\sim
\mathcal N(P_{\mathrm{mean}},P_{\mathrm{std}}^2)
}
$$

---

# 十二、全文总结

EDM 的核心并不是提出一种新的扩散模型，而是重新审视扩散生成模型中的各个设计变量。

在数学层面，EDM 将 VP、VE 等不同加噪过程统一到：

$$
\boxed{
X_t=s(t)[Y+\sigma(t)Z]
}
$$

并通过 Score 和 Probability Flow ODE 描述生成过程。

在采样层面，EDM 采用：

$$
\boxed{
\text{简单 ODE 参数化}
+
\text{非均匀噪声网格}
+
\text{Heun}
}
$$

并通过 Stochastic Churn 研究随机性对生成质量的影响。

在训练层面，EDM 将原始 Denoiser：

$$
D_\theta(X;\sigma)
$$

重新参数化为：

$$
\boxed{
D_\theta
=
c_{\mathrm{skip}}X
+
c_{\mathrm{out}}F_\theta
}
$$

使网络学习尺度稳定的残差目标，并通过：

$$
\boxed{
\lambda c_{\mathrm{out}}^2=1
}
$$

让内部网络使用统一尺度的 MSE。

同时，EDM 通过对数正态噪声分布调整不同噪声区域的训练资源分配。

因此，这篇论文最值得理解的不是某一个具体公式，而是它的整体研究方法：

$$
\boxed{
\text{将生成模型拆分为相对独立的模块，分别分析数学性质、数值误差和训练优化条件，再通过消融实验确定有效组合。}
}
$$

对于自动驾驶轨迹规划，这种研究思路同样具有价值。特别是**采样器设计、网络预测空间、Loss 空间和训练时间分布**，都可以作为独立变量进行分析，而不必将它们与某一种固定的 Flow Matching 或 Diffusion 架构绑定。

---

**参考资料**

- [EDM 原论文（NeurIPS 2022）](https://papers.nips.cc/paper_files/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf)
- [arXiv: 2206.00364](https://arxiv.org/abs/2206.00364)
- [NVIDIA 官方代码](https://github.com/NVlabs/edm)
